In [ ]:
"""
TASK 3 — Synapse-to-tracheole proximity and neurotransmitter analysis.

Inputs
------
syn_csv
    Princeton FAFB v783 synapse table containing presynaptic root IDs,
    presynaptic coordinates, and neuropil assignments.

nt_csv
    FlyWire neuron table containing neurotransmitter identity and prediction
    score for each neuronal root.

trach_csv
    CATMAID node table for the 117 reconstructed tracheoles, containing
    skeleton IDs, node coordinates, and parent-child relationships.

Analysis
--------
1. Neurotransmitter annotation
   syn_csv + nt_csv -> nt_dir

   Presynaptic FlyWire root IDs in the synapse table are matched to the
   neurotransmitter table. This generates the neurotransmitter lookup and
   summary tables used for downstream synapse analyses.

2. Synapse-to-tracheole distance
   syn_csv + nt_csv + trach_csv -> dist_dir

   CATMAID tracheole skeleton edges are resampled at 100 nm spacing and used
   to construct the tracheole reference point cloud. For each presynaptic
   contact, the distance from its presynaptic coordinate to the nearest
   tracheole skeleton point is calculated. Synapses are annotated by
   neuropil and neurotransmitter class.

   Outputs include:
   - distance summaries by neuropil
   - distance summaries by neuropil and neurotransmitter
   - distance histograms
   - sampled contact-level data for violin plots

3. Main statistical summaries and figures
   dist_dir -> final_dir / FINAL_TABLES
            -> final_dir / FINAL_FIGURES / Figures 01-08

   LA_L and LA_R are excluded. Neuropils are also grouped into super-regions.
   Distance distributions are summarized at neuropil, super-region, and
   neurotransmitter levels, and the corresponding statistical comparison
   tables are generated.

   Figures 01-08 contain:
   - neuropil distance distributions
   - neuropil median and IQR summaries
   - super-region distance distributions
   - super-region median and IQR summaries
   - neurotransmitter-specific neuropil distributions
   - neurotransmitter-specific super-region distributions
   - neuropil × neurotransmitter median-distance heatmap
   - super-region × neurotransmitter median-distance heatmap

4. Effect-size analysis
   final pairwise-comparison table -> effect_dir

   Pairwise distance comparisons are converted into effect-size summaries
   for reporting and downstream interpretation.

5. Additional summary figures
   final tables + distance histograms
   -> final_dir / FINAL_FIGURES / Figures 09-12

   Figures 09-12 summarize:
   - super-region proximity and ranked regional differences
   - neurotransmitter × super-region proximity patterns
   - cumulative distance coverage and r50/r90 distances
   - neurotransmitter proximity fingerprints and clustering

Final outputs
-------------
final_dir / FINAL_TABLES
    Statistical and numerical summary tables.

final_dir / FINAL_FIGURES
    Final publication figures 01-12.

effect_dir
    Pairwise effect-size tables.
"""

from pathlib import Path
from typing import Iterable
from collections import Counter, defaultdict
import itertools
import json
import math
import re
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import patches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.spatial import cKDTree
from scipy.stats import kruskal, mannwhitneyu
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import pdist

# Paths
downloads = Path.home() / "Downloads"
syn_csv = downloads / "fafb_v783_princeton_synapse_table.csv"
nt_csv = downloads / "neurons.csv"
trach_csv = downloads / "catmaid_nodes_all_117_tracheoles.csv.gz"
nt_dir = downloads / "task3_princeton_synapse_nt_join"
dist_dir = downloads / "task3_synapse_to_trachea_proximity"
final_dir = downloads / "task3_FINAL_summary_AZ_exclude_LA"
effect_dir = downloads / "task3_median_difference_effect_size_plots_EXCLUDE_LA"
fig_dir = final_dir / "FINAL_FIGURES"

# Analysis settings
dist_col = "distance_to_nearest_trachea_um"
nt_order = ["ACH", "GABA", "GLUT", "DA", "SER", "OCT"]
excluded_regions = {"LA_L", "LA_R"}
join_chunk = 500_000
nt_min_score = 0.5
join_sample_n = 10_000
dist_chunk = 250_000
workers = -1
trach_step_nm = 100.0
min_trach_nodes = 101
coord_mode = "auto"
hist_max_um = 100.0
hist_bin_um = 0.25
sample_region_n = 5_000
sample_region_nt_n = 2_000
sample_overshoot = 1.25
seed = 20260806
groups_per_page = 20
min_sample_n = 20
sample_super_n = 10_000
sample_super_nt_n = 5_000
plot_qmax = 0.995
top_effects_n = 20
effect_top_n = 25
effect_label_n = 12
effect_sig_only = False
min_abs_effect = 0.0
min_abs_median_um = 0.0
schematic_metric = "median_distance"
sig_outline_threshold = 0.50
coverage_um = [1, 2, 5, 10, 20]
coverage_xmax = 20.0

# Column names and super-region definitions
syn_root_cols = [
    "pre_root_id_720575940",
    "pre_root_id",
    "presynaptic_root_id",
    "pre_pt_root_id",
]

region_cols = [
    "neuropil",
    "brain_region",
    "region",
    "roi",
]

size_cols = [
    "size",
    "synapse_size",
    "cleft_size",
]

nt_root_cols = [
    "root_id",
    "root",
    "flywire_root_id",
    "segment_id",
]

nt_type_cols = [
    "nt_type",
    "transmitter",
    "neurotransmitter",
]

nt_score_cols = [
    "nt_type_score",
    "transmitter_prob",
    "transmitter_probability",
    "nt_score",
]

nt_group_cols = [
    "group",
    "cell_group",
]

nt_prob_cols = [
    "da_avg",
    "ser_avg",
    "gaba_avg",
    "glut_avg",
    "ach_avg",
    "oct_avg",
]

sample_coord_cols = [
    "pre_x",
    "pre_y",
    "pre_z",
    "ctr_x",
    "ctr_y",
    "ctr_z",
    "post_x",
    "post_y",
    "post_z",
]

syn_root_cols_b = [
    "pre_root_id_720575940",
    "pre_root_id",
    "presynaptic_root_id",
    "pre_pt_root_id",
]

region_cols_b = [
    "neuropil",
    "brain_region",
    "region",
    "roi",
]

x_cols = [
    "pre_x",
    "presynaptic_x",
    "pre_pt_position_x",
]

y_cols = [
    "pre_y",
    "presynaptic_y",
    "pre_pt_position_y",
]

z_cols = [
    "pre_z",
    "presynaptic_z",
    "pre_pt_position_z",
]

trach_sid_cols = [
    "skeleton_id",
    "new_skeleton_id",
    "skid",
    "neuron",
]

trach_node_cols = [
    "node_id",
    "treenode_id",
    "id",
]

trach_parent_cols = [
    "parent_id",
    "parent_node_id",
    "parent",
]

trach_x_cols = [
    "x_nm",
    "x",
]

trach_y_cols = [
    "y_nm",
    "y",
]

trach_z_cols = [
    "z_nm",
    "z",
]

superregions = {
    "OL": ["ME_L", "ME_R", "AME_L", "AME_R", "LO_L", "LO_R", "LOP_L", "LOP_R"],
    "MB": ["MB_CA_R", "MB_CA_L", "MB_PED_L", "MB_PED_R", "MB_VL_R", "MB_VL_L", "MB_ML_R", "MB_ML_L"],
    "CX": ["FB", "EB", "PB", "NO"],
    "LX": ["BU_L", "BU_R", "LAL_R", "LAL_L", "GA_R", "GA_L"],
    "VLNP": ["AOTU_R", "AOTU_L", "PVLP_L", "AVLP_L", "PVLP_R", "AVLP_R", "PLP_L", "PLP_R", "WED_L", "WED_R"],
    "LH": ["LH_L", "LH_R"],
    "SNP": ["SLP_L", "SLP_R", "SIP_L", "SIP_R", "SMP_R", "SMP_L"],
    "INP": ["CRE_L", "CRE_R", "ICL_L", "SCL_L", "ICL_R", "SCL_R", "IB_L", "IB_R", "ATL_R", "ATL_L"],
    "AL": ["AL_L", "AL_R"],
    "VMNP": ["VES_L", "VES_R", "EPA_R", "EPA_L", "GOR_L", "GOR_R", "SPS_L", "SPS_R", "IPS_R", "IPS_L"],
    "PENP": ["SAD", "AMMC_R", "AMMC_L", "FLA_R", "FLA_L", "CAN_R", "CAN_L", "PRW"],
    "GNG": ["GNG"],
    "OCG": ["OCG"],
}

super_order = list(superregions)
region_to_super = {
    neuropil: super_region
    for super_region, neuropils in superregions.items()
    for neuropil in neuropils
}

super_pos = {
    "OL": (0.07, 0.72), "VLNP": (0.22, 0.58), "LH": (0.31, 0.66),
    "MB": (0.42, 0.72), "CX": (0.50, 0.56), "LX": (0.36, 0.50),
    "SNP": (0.60, 0.67), "INP": (0.71, 0.56), "AL": (0.23, 0.35),
    "VMNP": (0.52, 0.36), "PENP": (0.72, 0.36), "GNG": (0.50, 0.18),
    "OCG": (0.86, 0.18),
}

def normalize_name(value: object) -> str:
    return re.sub('[^a-z0-9]+', '_', str(value).strip().lower()).strip('_')

def resolve_column(columns: Iterable[str], candidates: Iterable[str], *, required: bool=True) -> str | None:
    lookup = {normalize_name(column): column for column in columns}
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in lookup:
            return lookup[key]
    return None

def clean_root_id(series: pd.Series) -> pd.Series:
    cleaned = series.astype('string').str.strip().str.replace('\\.0$', '', regex=True).replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, '<NA>': pd.NA})
    return cleaned

def clean_text(series: pd.Series) -> pd.Series:
    return series.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, '<NA>': pd.NA})

def require_file(path: Path) -> Path:
    path = Path(path).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(f"Required file not found:\n{path}")
    return path

# Contact summaries
class SynapseSizeMoments:

    def __init__(self) -> None:
        self.n = 0
        self.total = 0.0
        self.mean = 0.0
        self.m2 = 0.0
        self.minimum: float | None = None
        self.maximum: float | None = None

    def update(self, values: pd.Series) -> None:
        array = pd.to_numeric(values, errors='coerce').dropna().to_numpy(dtype=np.float64)
        if len(array) == 0:
            return
        chunk_n = int(len(array))
        chunk_mean = float(array.mean())
        chunk_m2 = float(np.square(array - chunk_mean).sum())
        if self.n == 0:
            self.n = chunk_n
            self.mean = chunk_mean
            self.m2 = chunk_m2
        else:
            combined_n = self.n + chunk_n
            delta = chunk_mean - self.mean
            self.mean = self.mean + delta * chunk_n / combined_n
            self.m2 = self.m2 + chunk_m2 + delta * delta * self.n * chunk_n / combined_n
            self.n = combined_n
        self.total += float(array.sum())
        chunk_min = float(array.min())
        chunk_max = float(array.max())
        self.minimum = chunk_min if self.minimum is None else min(self.minimum, chunk_min)
        self.maximum = chunk_max if self.maximum is None else max(self.maximum, chunk_max)

    def as_dict(self) -> dict:
        standard_deviation = math.sqrt(self.m2 / (self.n - 1)) if self.n > 1 else float('nan')
        return {'n_non_null': int(self.n), 'sum': float(self.total), 'mean': float(self.mean) if self.n else float('nan'), 'standard_deviation': float(standard_deviation), 'minimum': self.minimum, 'maximum': self.maximum}

def load_nt_table(path: Path, output_dir: Path) -> tuple[pd.DataFrame, dict]:
    header = pd.read_csv(path, nrows=0)
    columns = list(header.columns)
    root_column = resolve_column(columns, nt_root_cols)
    type_column = resolve_column(columns, nt_type_cols)
    score_column = resolve_column(columns, nt_score_cols)
    group_column = resolve_column(columns, nt_group_cols, required=False)
    component_columns = [column for column in nt_prob_cols if column in columns]
    usecols = [root_column, type_column, score_column]
    if group_column is not None:
        usecols.append(group_column)
    usecols.extend(component_columns)
    nt = pd.read_csv(path, usecols=usecols, dtype={root_column: 'string'}, low_memory=False)
    rename_map = {root_column: 'root_id', type_column: 'nt_type', score_column: 'nt_type_score'}
    if group_column is not None:
        rename_map[group_column] = 'group'
    nt = nt.rename(columns=rename_map)
    nt['root_id'] = clean_root_id(nt['root_id'])
    nt['nt_type'] = clean_text(nt['nt_type'])
    nt['nt_type_score'] = pd.to_numeric(nt['nt_type_score'], errors='coerce')
    for column in component_columns:
        nt[column] = pd.to_numeric(nt[column], errors='coerce')
    nt = nt.dropna(subset=['root_id']).copy()
    duplicate_mask = nt.duplicated(subset=['root_id'], keep=False)
    duplicate_count = int(duplicate_mask.sum())
    duplicate_root_count = int(nt.loc[duplicate_mask, 'root_id'].nunique())
    duplicate_path = None
    if duplicate_count:
        duplicate_path = output_dir / 'duplicate_nt_root_rows.csv'
        nt.loc[duplicate_mask].sort_values(['root_id', 'nt_type_score'], ascending=[True, False]).to_csv(duplicate_path, index=False)
        nt = nt.sort_values(['root_id', 'nt_type_score'], ascending=[True, False], na_position='last').drop_duplicates(subset=['root_id'], keep='first').reset_index(drop=True)
    clean_columns = ['root_id', 'nt_type', 'nt_type_score']
    if 'group' in nt.columns:
        clean_columns.append('group')
    clean_columns.extend(component_columns)
    nt = nt[clean_columns].copy()
    clean_path = output_dir / 'nt_lookup_clean.csv.gz'
    nt.to_csv(clean_path, index=False, compression='gzip')
    cell_counts = nt.assign(nt_type_for_count=nt['nt_type'].fillna('<MISSING>')).groupby('nt_type_for_count', dropna=False).agg(cell_count=('root_id', 'size'), median_nt_type_score=('nt_type_score', 'median'), mean_nt_type_score=('nt_type_score', 'mean')).reset_index().rename(columns={'nt_type_for_count': 'nt_type'}).sort_values('cell_count', ascending=False)
    cell_counts.to_csv(output_dir / 'neurons_nt_cell_counts.csv', index=False)
    metadata = {'input_rows': int(len(pd.read_csv(path, usecols=[root_column]))), 'clean_unique_root_rows': int(len(nt)), 'duplicate_rows': duplicate_count, 'duplicate_root_ids': duplicate_root_count, 'duplicate_rows_file': str(duplicate_path) if duplicate_path is not None else None, 'resolved_columns': {'root_id': root_column, 'nt_type': type_column, 'nt_type_score': score_column, 'group': group_column, 'component_scores': component_columns}, 'clean_lookup_file': str(clean_path)}
    return (nt, metadata)

def update_counter_from_series(counter: Counter, series: pd.Series) -> None:
    counter.update(series.value_counts(dropna=False).to_dict())

def join_nt(synapse_csv, neurons_csv, output_dir, chunksize, nt_threshold, sample_rows) -> int:
    synapse_path = synapse_csv.expanduser().resolve()
    neuron_path = neurons_csv.expanduser().resolve()
    output_dir = output_dir.expanduser().resolve()
    if not synapse_path.exists():
        raise FileNotFoundError(f'Missing synapse CSV:\n{synapse_path}')
    if not neuron_path.exists():
        raise FileNotFoundError(f'Missing neuron/NT CSV:\n{neuron_path}')
    if chunksize <= 0:
        raise ValueError('--chunksize must be positive.')
    if not 0 <= nt_threshold <= 1:
        raise ValueError('--nt-threshold must be between 0 and 1.')
    output_dir.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    nt, nt_metadata = load_nt_table(neuron_path, output_dir)
    synapse_header = pd.read_csv(synapse_path, nrows=0)
    synapse_columns = list(synapse_header.columns)
    pre_root_column = resolve_column(synapse_columns, syn_root_cols)
    neuropil_column = resolve_column(synapse_columns, region_cols)
    size_column = resolve_column(synapse_columns, size_cols, required=False)
    sample_coordinate_columns = [column for column in sample_coord_cols if column in synapse_columns]
    usecols = [pre_root_column, neuropil_column]
    if size_column is not None:
        usecols.append(size_column)
    usecols.extend(sample_coordinate_columns)
    usecols = list(dict.fromkeys(usecols))
    nt_join_columns = ['root_id', 'nt_type', 'nt_type_score']
    if 'group' in nt.columns:
        nt_join_columns.append('group')
    nt_join_columns.extend([column for column in nt_prob_cols if column in nt.columns])
    nt_lookup = nt[nt_join_columns].copy()
    all_contacts = Counter()
    matched_contacts = Counter()
    labelled_contacts = Counter()
    high_conf_contacts = Counter()
    neuropil_nt_all = Counter()
    neuropil_nt_high_conf = Counter()
    global_nt_all = Counter()
    global_nt_high_conf = Counter()
    unique_roots_by_neuropil: dict[str, set[str]] = defaultdict(set)
    all_pre_roots: set[str] = set()
    size_by_neuropil: dict[str, SynapseSizeMoments] = defaultdict(SynapseSizeMoments)
    sample_parts: list[pd.DataFrame] = []
    sample_remaining = max(int(sample_rows), 0)
    total_rows = 0
    matched_rows = 0
    labelled_rows = 0
    high_conf_rows = 0
    missing_pre_root_rows = 0
    reader = pd.read_csv(synapse_path, usecols=usecols, dtype={pre_root_column: 'string', neuropil_column: 'string'}, chunksize=chunksize, low_memory=False)
    for chunk_index, chunk in enumerate(reader, start=1):
        if chunk.empty:
            break
        chunk = chunk.rename(columns={pre_root_column: 'pre_root_id', neuropil_column: 'neuropil'})
        if size_column is not None:
            chunk = chunk.rename(columns={size_column: 'synapse_size'})
        chunk['pre_root_id'] = clean_root_id(chunk['pre_root_id'])
        chunk['neuropil'] = clean_text(chunk['neuropil']).fillna('<MISSING>')
        missing_pre_root_rows += int(chunk['pre_root_id'].isna().sum())
        valid_roots = chunk['pre_root_id'].dropna().astype(str)
        all_pre_roots.update(valid_roots.unique())
        root_membership = chunk.loc[chunk['pre_root_id'].notna(), ['neuropil', 'pre_root_id']].drop_duplicates()
        for neuropil, roots in root_membership.groupby('neuropil', sort=False)['pre_root_id']:
            unique_roots_by_neuropil[str(neuropil)].update(roots.astype(str))
        merged = chunk.merge(nt_lookup, how='left', left_on='pre_root_id', right_on='root_id', validate='many_to_one', indicator=True)
        matched_mask = merged['_merge'] == 'both'
        labelled_mask = matched_mask & merged['nt_type'].notna()
        high_conf_mask = labelled_mask & merged['nt_type_score'].ge(nt_threshold)
        chunk_rows = int(len(merged))
        chunk_matched = int(matched_mask.sum())
        chunk_labelled = int(labelled_mask.sum())
        chunk_high_conf = int(high_conf_mask.sum())
        total_rows += chunk_rows
        matched_rows += chunk_matched
        labelled_rows += chunk_labelled
        high_conf_rows += chunk_high_conf
        update_counter_from_series(all_contacts, merged['neuropil'])
        update_counter_from_series(matched_contacts, merged.loc[matched_mask, 'neuropil'])
        update_counter_from_series(labelled_contacts, merged.loc[labelled_mask, 'neuropil'])
        update_counter_from_series(high_conf_contacts, merged.loc[high_conf_mask, 'neuropil'])
        labelled_group = merged.loc[labelled_mask, ['neuropil', 'nt_type']].value_counts()
        for (neuropil, nt_type), count in labelled_group.items():
            neuropil_nt_all[str(neuropil), str(nt_type)] += int(count)
            global_nt_all[str(nt_type)] += int(count)
        high_conf_group = merged.loc[high_conf_mask, ['neuropil', 'nt_type']].value_counts()
        for (neuropil, nt_type), count in high_conf_group.items():
            neuropil_nt_high_conf[str(neuropil), str(nt_type)] += int(count)
            global_nt_high_conf[str(nt_type)] += int(count)
        if size_column is not None and 'synapse_size' in merged.columns:
            for neuropil, values in merged.groupby('neuropil', sort=False)['synapse_size']:
                size_by_neuropil[str(neuropil)].update(values)
        if sample_remaining > 0:
            high_sample = merged.loc[high_conf_mask].head(min(sample_remaining, 250))
            if len(high_sample):
                sample_parts.append(high_sample.copy())
                sample_remaining -= int(len(high_sample))
            if sample_remaining > 0:
                general_sample = merged.head(min(sample_remaining, 250))
                sample_parts.append(general_sample.copy())
                sample_remaining -= int(len(general_sample))
        elapsed = time.perf_counter() - started
        rate = total_rows / elapsed if elapsed > 0 else float('nan')
    elapsed = time.perf_counter() - started
    nt_root_set = set(nt['root_id'].astype(str))
    matched_unique_roots = all_pre_roots & nt_root_set
    unmatched_unique_roots = all_pre_roots - nt_root_set
    neuropil_rows = []
    for neuropil, count in all_contacts.most_common():
        matched = int(matched_contacts[neuropil])
        labelled = int(labelled_contacts[neuropil])
        high_conf = int(high_conf_contacts[neuropil])
        unique_roots = len(unique_roots_by_neuropil[str(neuropil)])
        row = {'neuropil': neuropil, 'contact_row_count': int(count), 'fraction_of_all_contact_rows': count / total_rows if total_rows else float('nan'), 'unique_presynaptic_root_count': int(unique_roots), 'nt_root_matched_contact_count': matched, 'nt_root_match_fraction': matched / count if count else float('nan'), 'nt_labelled_contact_count': labelled, 'nt_labelled_fraction_of_all': labelled / count if count else float('nan'), 'high_conf_nt_contact_count': high_conf, 'high_conf_nt_fraction_of_all': high_conf / count if count else float('nan'), 'high_conf_fraction_of_labelled': high_conf / labelled if labelled else float('nan')}
        if neuropil in size_by_neuropil:
            size_summary = size_by_neuropil[neuropil].as_dict()
            for key, value in size_summary.items():
                row[f'synapse_size_{key}'] = value
        neuropil_rows.append(row)
    neuropil_summary = pd.DataFrame(neuropil_rows).sort_values(['contact_row_count', 'neuropil'], ascending=[False, True], kind='mergesort').reset_index(drop=True)
    nt_keys = sorted(set(neuropil_nt_all.keys()) | set(neuropil_nt_high_conf.keys()))
    neuropil_nt_rows = []
    for neuropil, nt_type in nt_keys:
        labelled_count = int(neuropil_nt_all[neuropil, nt_type])
        high_conf_count = int(neuropil_nt_high_conf[neuropil, nt_type])
        neuropil_total = int(all_contacts[neuropil])
        neuropil_labelled_total = int(labelled_contacts[neuropil])
        neuropil_high_conf_total = int(high_conf_contacts[neuropil])
        neuropil_nt_rows.append({'neuropil': neuropil, 'nt_type': nt_type, 'labelled_contact_count': labelled_count, 'high_conf_contact_count': high_conf_count, 'labelled_nt_fraction_of_all_contacts': labelled_count / neuropil_total if neuropil_total else float('nan'), 'labelled_nt_fraction_within_labelled_contacts': labelled_count / neuropil_labelled_total if neuropil_labelled_total else float('nan'), 'high_conf_nt_fraction_of_all_contacts': high_conf_count / neuropil_total if neuropil_total else float('nan'), 'high_conf_nt_fraction_within_high_conf_contacts': high_conf_count / neuropil_high_conf_total if neuropil_high_conf_total else float('nan'), 'nt_type_score_threshold': nt_threshold})
    neuropil_nt_df = pd.DataFrame(neuropil_nt_rows).sort_values(['neuropil', 'high_conf_contact_count', 'nt_type'], ascending=[True, False, True], kind='mergesort').reset_index(drop=True)
    global_nt_types = sorted(set(global_nt_all.keys()) | set(global_nt_high_conf.keys()))
    global_nt_df = pd.DataFrame([{'nt_type': nt_type, 'labelled_contact_count': int(global_nt_all[nt_type]), 'high_conf_contact_count': int(global_nt_high_conf[nt_type]), 'labelled_fraction_of_all_contacts': global_nt_all[nt_type] / total_rows if total_rows else float('nan'), 'high_conf_fraction_of_all_contacts': global_nt_high_conf[nt_type] / total_rows if total_rows else float('nan'), 'nt_type_score_threshold': nt_threshold} for nt_type in global_nt_types]).sort_values('high_conf_contact_count', ascending=False)
    global_summary = pd.DataFrame([{'metric': 'synapse_contact_rows_processed', 'value': total_rows, 'unit': 'rows'}, {'metric': 'rows_with_missing_pre_root_id', 'value': missing_pre_root_rows, 'unit': 'rows'}, {'metric': 'nt_root_matched_contact_rows', 'value': matched_rows, 'unit': 'rows'}, {'metric': 'nt_root_match_fraction_of_rows', 'value': matched_rows / total_rows if total_rows else float('nan'), 'unit': 'fraction'}, {'metric': 'nt_labelled_contact_rows', 'value': labelled_rows, 'unit': 'rows'}, {'metric': 'nt_labelled_fraction_of_rows', 'value': labelled_rows / total_rows if total_rows else float('nan'), 'unit': 'fraction'}, {'metric': 'high_conf_nt_contact_rows', 'value': high_conf_rows, 'unit': 'rows'}, {'metric': 'high_conf_nt_fraction_of_rows', 'value': high_conf_rows / total_rows if total_rows else float('nan'), 'unit': 'fraction'}, {'metric': 'unique_presynaptic_roots_in_synapse_rows', 'value': len(all_pre_roots), 'unit': 'roots'}, {'metric': 'unique_presynaptic_roots_matched_to_nt_table', 'value': len(matched_unique_roots), 'unit': 'roots'}, {'metric': 'unique_presynaptic_roots_unmatched_to_nt_table', 'value': len(unmatched_unique_roots), 'unit': 'roots'}, {'metric': 'unique_root_level_match_fraction', 'value': len(matched_unique_roots) / len(all_pre_roots) if all_pre_roots else float('nan'), 'unit': 'fraction'}, {'metric': 'neuropil_labels_observed', 'value': len(all_contacts), 'unit': 'labels'}, {'metric': 'nt_type_score_threshold', 'value': nt_threshold, 'unit': 'score'}])
    neuropil_summary_path = output_dir / 'neuropil_contact_summary.csv'
    neuropil_nt_path = output_dir / 'neuropil_nt_contact_counts.csv'
    global_nt_path = output_dir / 'global_nt_contact_counts.csv'
    global_summary_path = output_dir / 'global_join_summary.csv'
    neuropil_summary.to_csv(neuropil_summary_path, index=False)
    neuropil_nt_df.to_csv(neuropil_nt_path, index=False)
    global_nt_df.to_csv(global_nt_path, index=False)
    global_summary.to_csv(global_summary_path, index=False)
    if sample_parts:
        joined_sample = pd.concat(sample_parts, ignore_index=True).head(sample_rows)
    else:
        joined_sample = pd.DataFrame()
    joined_sample_path = output_dir / 'joined_sample.csv.gz'
    joined_sample.to_csv(joined_sample_path, index=False, compression='gzip')
    return 0

# Synapse-to-tracheole distances
def header_prefix_from_column(column_name: str) -> str | None:
    match = re.search('(720575940)$', str(column_name))
    if match:
        return match.group(1)
    match = re.search('root_id_(\\d{9})', str(column_name))
    if match:
        return match.group(1)
    return None

def transform_synapse_root_ids(series: pd.Series, *, mode: str, prefix: str | None) -> pd.Series:
    cleaned = clean_root_id(series)
    if mode == 'raw':
        return cleaned
    if mode == 'header_prefix':
        if not prefix:
            raise ValueError('header_prefix mode requires a prefix.')
        full_length = 18
        suffix_length = full_length - len(prefix)
        result = cleaned.copy()
        short_mask = result.notna() & result.str.fullmatch(f'\\d{{1,{suffix_length}}}')
        result.loc[short_mask] = prefix + result.loc[short_mask].str.zfill(suffix_length)
        return result
    raise ValueError(f'Unsupported root-ID mode: {mode}')

def select_root_id_encoding(synapse_path: Path, pre_root_column: str, nt_root_ids: set[str]) -> dict:
    probe = pd.read_csv(synapse_path, usecols=[pre_root_column], dtype={pre_root_column: 'string'}, nrows=20000)[pre_root_column]
    candidates = [{'mode': 'raw', 'prefix': None}]
    prefix = header_prefix_from_column(pre_root_column)
    if prefix:
        candidates.append({'mode': 'header_prefix', 'prefix': prefix})
    diagnostics = []
    for candidate in candidates:
        transformed = transform_synapse_root_ids(probe, mode=candidate['mode'], prefix=candidate['prefix'])
        unique = set(transformed.dropna().astype(str))
        matched = unique & nt_root_ids
        diagnostics.append({**candidate, 'n_unique_probe_roots': len(unique), 'n_unique_matches': len(matched), 'unique_match_fraction': len(matched) / len(unique) if unique else 0.0, 'examples_raw': clean_root_id(probe).dropna().head(5).tolist(), 'examples_transformed': transformed.dropna().head(5).tolist()})
    best = max(diagnostics, key=lambda row: row['unique_match_fraction'])
    if best['unique_match_fraction'] <= 0:
        raise RuntimeError('Could not reconcile Princeton root IDs with neurons.csv.\n' + json.dumps(diagnostics, indent=2))
    return {'selected': {'mode': best['mode'], 'prefix': best['prefix']}, 'diagnostics': diagnostics}

def load_nt_lookup(path: Path) -> tuple[pd.DataFrame, dict]:
    header = pd.read_csv(path, nrows=0)
    columns = list(header.columns)
    root_column = resolve_column(columns, nt_root_cols)
    type_column = resolve_column(columns, nt_type_cols)
    score_column = resolve_column(columns, nt_score_cols)
    table = pd.read_csv(path, usecols=[root_column, type_column, score_column], dtype={root_column: 'string'}, low_memory=False).rename(columns={root_column: 'root_id', type_column: 'nt_type', score_column: 'nt_type_score'})
    table['root_id'] = clean_root_id(table['root_id'])
    table['nt_type'] = clean_text(table['nt_type'])
    table['nt_type_score'] = pd.to_numeric(table['nt_type_score'], errors='coerce')
    table = table.dropna(subset=['root_id']).copy()
    duplicate_mask = table.duplicated(subset=['root_id'], keep=False)
    if duplicate_mask.any():
        table = table.sort_values(['root_id', 'nt_type_score'], ascending=[True, False], na_position='last').drop_duplicates(subset=['root_id'], keep='first')
    table = table.reset_index(drop=True)
    metadata = {'root_column': root_column, 'type_column': type_column, 'score_column': score_column, 'unique_roots': int(len(table)), 'labelled_roots': int(table['nt_type'].notna().sum())}
    return (table, metadata)

def load_standardised_trachea_nodes(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path, low_memory=False)
    columns = list(raw.columns)
    skid_column = resolve_column(columns, trach_sid_cols)
    node_column = resolve_column(columns, trach_node_cols)
    parent_column = resolve_column(columns, trach_parent_cols)
    x_column = resolve_column(columns, trach_x_cols)
    y_column = resolve_column(columns, trach_y_cols)
    z_column = resolve_column(columns, trach_z_cols)
    nodes = pd.DataFrame({'skeleton_id': pd.to_numeric(raw[skid_column], errors='coerce'), 'node_id': pd.to_numeric(raw[node_column], errors='coerce'), 'parent_id': pd.to_numeric(raw[parent_column], errors='coerce'), 'x_nm': pd.to_numeric(raw[x_column], errors='coerce'), 'y_nm': pd.to_numeric(raw[y_column], errors='coerce'), 'z_nm': pd.to_numeric(raw[z_column], errors='coerce')}).dropna(subset=['skeleton_id', 'node_id', 'x_nm', 'y_nm', 'z_nm'])
    nodes['skeleton_id'] = nodes['skeleton_id'].astype(np.int64)
    nodes['node_id'] = nodes['node_id'].astype(np.int64)
    nodes['parent_id'] = pd.to_numeric(nodes['parent_id'], errors='coerce').astype('Int64')
    nodes = nodes.drop_duplicates(['skeleton_id', 'node_id']).reset_index(drop=True)
    return nodes

def filter_trachea_by_node_count(nodes: pd.DataFrame, minimum_original_nodes: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    counts = nodes.groupby('skeleton_id').size().rename('original_node_count').reset_index()
    counts['retained'] = counts['original_node_count'] >= minimum_original_nodes
    retained_ids = set(counts.loc[counts['retained'], 'skeleton_id'].astype(int))
    filtered = nodes[nodes['skeleton_id'].isin(retained_ids)].copy()
    if filtered.empty:
        raise RuntimeError('No tracheole skeletons remained after node-count filtering.')
    return (filtered, counts)

def sample_trachea_edges(nodes: pd.DataFrame, spacing_nm: float) -> tuple[np.ndarray, dict]:
    if spacing_nm <= 0:
        raise ValueError('resample spacing must be positive.')
    sampled_parts: list[np.ndarray] = []
    edge_count = 0
    total_length_nm = 0.0
    missing_parent_edges = 0
    for skeleton_id, group in nodes.groupby('skeleton_id', sort=False):
        node_ids = group['node_id'].to_numpy(dtype=np.int64)
        xyz = group[['x_nm', 'y_nm', 'z_nm']].to_numpy(dtype=np.float64)
        lookup = {int(node_id): xyz[index] for index, node_id in enumerate(node_ids)}
        roots = group[group['parent_id'].isna()]
        if len(roots):
            sampled_parts.append(roots[['x_nm', 'y_nm', 'z_nm']].to_numpy(dtype=np.float64))
        for row in group.itertuples(index=False):
            if pd.isna(row.parent_id):
                continue
            child = np.array([row.x_nm, row.y_nm, row.z_nm], dtype=np.float64)
            parent = lookup.get(int(row.parent_id))
            if parent is None:
                missing_parent_edges += 1
                continue
            vector = child - parent
            length = float(np.linalg.norm(vector))
            if not np.isfinite(length):
                continue
            edge_count += 1
            total_length_nm += length
            if length == 0:
                sampled_parts.append(child.reshape(1, 3))
                continue
            intervals = max(int(math.ceil(length / spacing_nm)), 1)
            fractions = np.arange(1, intervals + 1, dtype=np.float64) / intervals
            points = parent[None, :] + fractions[:, None] * vector[None, :]
            sampled_parts.append(points)
    if not sampled_parts:
        raise RuntimeError('No tracheole points were generated.')
    sampled = np.vstack(sampled_parts).astype(np.float32, copy=False)
    metadata = {'retained_skeleton_count': int(nodes['skeleton_id'].nunique()), 'retained_original_node_count': int(len(nodes)), 'sampled_point_count': int(len(sampled)), 'edge_count': int(edge_count), 'missing_parent_edges': int(missing_parent_edges), 'total_skeleton_length_nm': float(total_length_nm), 'total_skeleton_length_um': float(total_length_nm / 1000.0), 'resample_spacing_nm': float(spacing_nm)}
    return (sampled, metadata)

def apply_coordinate_transform(coordinates: np.ndarray, mode: str) -> np.ndarray:
    coordinates = np.asarray(coordinates, dtype=np.float64)
    if mode == 'nm':
        return coordinates
    if mode == 'voxel_4_4_40':
        return coordinates * np.array([4.0, 4.0, 40.0], dtype=np.float64)[None, :]
    raise ValueError(f'Unsupported coordinate mode: {mode}')

def select_coordinate_mode(synapse_path: Path, coordinate_columns: list[str], trachea_points_nm: np.ndarray, tree: cKDTree, requested_mode: str) -> dict:
    probe = pd.read_csv(synapse_path, usecols=coordinate_columns, nrows=20000, low_memory=False)
    probe_xyz = probe[coordinate_columns].apply(pd.to_numeric, errors='coerce').dropna().to_numpy(dtype=np.float64)
    if not len(probe_xyz):
        raise RuntimeError('The synapse coordinate probe contains no usable rows.')
    modes = [requested_mode] if requested_mode != 'auto' else ['nm', 'voxel_4_4_40']
    trachea_min = np.asarray(trachea_points_nm, dtype=np.float64).min(axis=0)
    trachea_max = np.asarray(trachea_points_nm, dtype=np.float64).max(axis=0)
    padding_nm = 50000.0
    diagnostics = []
    for mode in modes:
        transformed = apply_coordinate_transform(probe_xyz, mode)
        inside = np.all(transformed >= (trachea_min - padding_nm)[None, :], axis=1) & np.all(transformed <= (trachea_max + padding_nm)[None, :], axis=1)
        distances_nm, _ = tree.query(transformed.astype(np.float32), k=1, workers=-1)
        diagnostics.append({'mode': mode, 'probe_rows': int(len(transformed)), 'fraction_inside_trachea_bbox_plus_50um': float(inside.mean()), 'median_nearest_trachea_distance_um': float(np.median(distances_nm) / 1000.0), 'q95_nearest_trachea_distance_um': float(np.quantile(distances_nm, 0.95) / 1000.0), 'coordinate_min_nm': transformed.min(axis=0).tolist(), 'coordinate_max_nm': transformed.max(axis=0).tolist()})
    selected = min(diagnostics, key=lambda row: (-row['fraction_inside_trachea_bbox_plus_50um'], row['median_nearest_trachea_distance_um']))
    return {'selected_mode': selected['mode'], 'diagnostics': diagnostics, 'trachea_bbox_min_nm': trachea_min.tolist(), 'trachea_bbox_max_nm': trachea_max.tolist()}

class DistanceMoments:

    def __init__(self) -> None:
        self.n = 0
        self.total = 0.0
        self.mean = 0.0
        self.m2 = 0.0
        self.minimum: float | None = None
        self.maximum: float | None = None

    def update_array(self, values: np.ndarray) -> None:
        array = np.asarray(values, dtype=np.float64)
        array = array[np.isfinite(array)]
        if len(array) == 0:
            return
        chunk_n = int(len(array))
        chunk_mean = float(array.mean())
        chunk_m2 = float(np.square(array - chunk_mean).sum())
        if self.n == 0:
            self.n = chunk_n
            self.mean = chunk_mean
            self.m2 = chunk_m2
        else:
            combined = self.n + chunk_n
            delta = chunk_mean - self.mean
            self.mean = self.mean + delta * chunk_n / combined
            self.m2 = self.m2 + chunk_m2 + delta * delta * self.n * chunk_n / combined
            self.n = combined
        self.total += float(array.sum())
        chunk_min = float(array.min())
        chunk_max = float(array.max())
        self.minimum = chunk_min if self.minimum is None else min(self.minimum, chunk_min)
        self.maximum = chunk_max if self.maximum is None else max(self.maximum, chunk_max)

    def as_dict(self) -> dict:
        sd = math.sqrt(self.m2 / (self.n - 1)) if self.n > 1 else float('nan')
        return {'n': int(self.n), 'mean_distance_um': float(self.mean) if self.n else float('nan'), 'standard_deviation_um': float(sd), 'minimum_distance_um': self.minimum, 'maximum_distance_um': self.maximum}

def update_group_summaries(dataframe: pd.DataFrame, group_columns: list[str], value_column: str, summary_store: dict, histogram_store: dict, histogram_edges: np.ndarray) -> None:
    grouped = dataframe.groupby(group_columns, dropna=False, sort=False)
    for key, group in grouped:
        if not isinstance(key, tuple):
            key = (key,)
        clean_key = tuple((str(value) for value in key))
        values = pd.to_numeric(group[value_column], errors='coerce').dropna().to_numpy(dtype=np.float64)
        if not len(values):
            continue
        summary_store[clean_key].update_array(values)
        histogram_store[clean_key] += np.histogram(values, bins=histogram_edges)[0]

def load_count_maps(join_dir: Path) -> tuple[dict[str, int], dict[tuple[str, str], int]]:
    neuropil_path = join_dir / 'neuropil_contact_summary.csv'
    neuropil_nt_path = join_dir / 'neuropil_nt_contact_counts.csv'
    all_counts: dict[str, int] = {}
    nt_counts: dict[tuple[str, str], int] = {}
    if neuropil_path.exists():
        table = pd.read_csv(neuropil_path)
        if {'neuropil', 'contact_row_count'}.issubset(table.columns):
            all_counts = {str(row.neuropil): int(row.contact_row_count) for row in table.itertuples(index=False)}
    if neuropil_nt_path.exists():
        table = pd.read_csv(neuropil_nt_path)
        if {'neuropil', 'nt_type', 'high_conf_contact_count'}.issubset(table.columns):
            nt_counts = {(str(row.neuropil), str(row.nt_type)): int(row.high_conf_contact_count) for row in table.itertuples(index=False)}
    return (all_counts, nt_counts)

def sample_probability(count: int | None, target: int, overshoot: float) -> float:
    if target <= 0:
        return 0.0
    if count is None or count <= 0:
        return 0.0
    return min(1.0, target * overshoot / count)

def cap_group_samples(dataframe: pd.DataFrame, group_columns: list[str], target: int, random_seed: int) -> pd.DataFrame:
    if dataframe.empty:
        return dataframe
    rng = np.random.default_rng(random_seed)
    order = rng.permutation(len(dataframe))
    shuffled = dataframe.iloc[order]
    capped = shuffled.groupby(group_columns, dropna=False, sort=False, group_keys=False).head(target).reset_index(drop=True)
    return capped

def summary_store_to_frame(store: dict, key_names: list[str]) -> pd.DataFrame:
    rows = []
    for key, moments in store.items():
        row = {name: value for name, value in zip(key_names, key)}
        row.update(moments.as_dict())
        rows.append(row)
    columns = key_names + ['n', 'mean_distance_um', 'standard_deviation_um', 'minimum_distance_um', 'maximum_distance_um']
    if not rows:
        return pd.DataFrame(columns=columns)
    return pd.DataFrame(rows)[columns]

def histogram_store_to_frame(store: dict, key_names: list[str], edges: np.ndarray) -> pd.DataFrame:
    rows = []
    for key, counts in store.items():
        for index, count in enumerate(counts):
            row = {name: value for name, value in zip(key_names, key)}
            row.update({'bin_left_um': float(edges[index]), 'bin_right_um': float(edges[index + 1]), 'count': int(count)})
            rows.append(row)
    columns = key_names + ['bin_left_um', 'bin_right_um', 'count']
    if not rows:
        return pd.DataFrame(columns=columns)
    return pd.DataFrame(rows)[columns]

def measure_proximity(synapse_csv, neurons_csv, trachea_nodes, join_dir, output_dir, chunksize, query_workers, resample_spacing_nm, minimum_original_nodes, nt_threshold, coordinate_mode, histogram_max_um, histogram_bin_um, sample_per_neuropil, sample_per_neuropil_nt, sampling_overshoot, random_seed) -> int:
    synapse_path = synapse_csv.expanduser().resolve()
    neuron_path = neurons_csv.expanduser().resolve()
    join_dir = join_dir.expanduser().resolve()
    output_dir = output_dir.expanduser().resolve()
    if not synapse_path.exists():
        raise FileNotFoundError(f'Missing synapse table:\n{synapse_path}')
    if not neuron_path.exists():
        raise FileNotFoundError(f'Missing NT table:\n{neuron_path}')
    if chunksize <= 0:
        raise ValueError('--chunksize must be positive.')
    if histogram_max_um <= 0:
        raise ValueError('--histogram-max-um must be positive.')
    if histogram_bin_um <= 0:
        raise ValueError('--histogram-bin-um must be positive.')
    if not 0 <= nt_threshold <= 1:
        raise ValueError('--nt-threshold must be between 0 and 1.')
    output_dir.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    trachea_path = require_file(Path(trachea_nodes))
    nt_lookup, nt_metadata = load_nt_lookup(neuron_path)
    nt_root_ids = set(nt_lookup['root_id'].astype(str))
    trachea_nodes = load_standardised_trachea_nodes(trachea_path)
    filtered_nodes, skeleton_qc = filter_trachea_by_node_count(trachea_nodes, minimum_original_nodes)
    skeleton_qc_path = output_dir / 'trachea_skeleton_qc.csv'
    skeleton_qc.to_csv(skeleton_qc_path, index=False)
    sampled_trachea_nm, trachea_metadata = sample_trachea_edges(filtered_nodes, resample_spacing_nm)
    sampled_path = output_dir / f'sampled_trachea_points_{resample_spacing_nm:g}nm.npz'
    np.savez_compressed(sampled_path, points_nm=sampled_trachea_nm, spacing_nm=np.array([resample_spacing_nm], dtype=np.float64))
    tree = cKDTree(sampled_trachea_nm)
    synapse_header = pd.read_csv(synapse_path, nrows=0)
    synapse_columns = list(synapse_header.columns)
    pre_root_column = resolve_column(synapse_columns, syn_root_cols_b)
    neuropil_column = resolve_column(synapse_columns, region_cols_b)
    pre_x_column = resolve_column(synapse_columns, x_cols)
    pre_y_column = resolve_column(synapse_columns, y_cols)
    pre_z_column = resolve_column(synapse_columns, z_cols)
    coordinate_columns = [pre_x_column, pre_y_column, pre_z_column]
    root_encoding = select_root_id_encoding(synapse_path, pre_root_column, nt_root_ids)
    root_encoding_path = output_dir / 'root_id_transform_diagnostic.json'
    root_encoding_path.write_text(json.dumps(root_encoding, indent=2), encoding='utf-8')
    coordinate_diagnostic = select_coordinate_mode(synapse_path, coordinate_columns, sampled_trachea_nm, tree, coordinate_mode)
    coordinate_diagnostic_path = output_dir / 'coordinate_transform_diagnostic.json'
    coordinate_diagnostic_path.write_text(json.dumps(coordinate_diagnostic, indent=2), encoding='utf-8')
    selected_coordinate_mode = coordinate_diagnostic['selected_mode']
    all_count_map, nt_count_map = load_count_maps(join_dir)
    regular_edges = np.arange(0.0, histogram_max_um + histogram_bin_um, histogram_bin_um, dtype=np.float64)
    if regular_edges[-1] < histogram_max_um:
        regular_edges = np.append(regular_edges, histogram_max_um)
    histogram_edges = np.append(regular_edges, np.inf)
    overall_summary = defaultdict(DistanceMoments)
    neuropil_summary = defaultdict(DistanceMoments)
    neuropil_nt_summary = defaultdict(DistanceMoments)
    overall_hist = defaultdict(lambda: np.zeros(len(histogram_edges) - 1, dtype=np.int64))
    neuropil_hist = defaultdict(lambda: np.zeros(len(histogram_edges) - 1, dtype=np.int64))
    neuropil_nt_hist = defaultdict(lambda: np.zeros(len(histogram_edges) - 1, dtype=np.int64))
    all_sample_parts: list[pd.DataFrame] = []
    nt_sample_parts: list[pd.DataFrame] = []
    rng = np.random.default_rng(random_seed)
    total_rows = 0
    usable_coordinate_rows = 0
    nt_root_match_rows = 0
    labelled_nt_rows = 0
    high_conf_nt_rows = 0
    usecols = [pre_root_column, neuropil_column, pre_x_column, pre_y_column, pre_z_column]
    reader = pd.read_csv(synapse_path, usecols=usecols, dtype={pre_root_column: 'string', neuropil_column: 'string'}, chunksize=chunksize, low_memory=False)
    for chunk_index, chunk in enumerate(reader, start=1):
        if chunk.empty:
            break
        chunk = chunk.rename(columns={pre_root_column: 'pre_root_id_raw', neuropil_column: 'neuropil', pre_x_column: 'pre_x_raw', pre_y_column: 'pre_y_raw', pre_z_column: 'pre_z_raw'})
        chunk['neuropil'] = clean_text(chunk['neuropil']).fillna('<MISSING>')
        chunk['pre_root_id'] = transform_synapse_root_ids(chunk['pre_root_id_raw'], mode=root_encoding['selected']['mode'], prefix=root_encoding['selected']['prefix'])
        raw_coordinates = chunk[['pre_x_raw', 'pre_y_raw', 'pre_z_raw']].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=np.float64)
        coordinate_valid = np.all(np.isfinite(raw_coordinates), axis=1)
        distances_um = np.full(len(chunk), np.nan, dtype=np.float64)
        if coordinate_valid.any():
            transformed_nm = apply_coordinate_transform(raw_coordinates[coordinate_valid], selected_coordinate_mode)
            query_distance_nm, _ = tree.query(transformed_nm.astype(np.float32), k=1, workers=query_workers)
            distances_um[coordinate_valid] = query_distance_nm / 1000.0
        chunk['distance_to_nearest_trachea_um'] = distances_um
        merged = chunk.merge(nt_lookup, how='left', left_on='pre_root_id', right_on='root_id', validate='many_to_one', indicator=True)
        matched_mask = merged['_merge'] == 'both'
        labelled_mask = matched_mask & merged['nt_type'].notna()
        high_conf_mask = labelled_mask & merged['nt_type_score'].ge(nt_threshold)
        usable_mask = merged['distance_to_nearest_trachea_um'].notna()
        high_conf_usable = high_conf_mask & usable_mask
        chunk_rows = int(len(merged))
        total_rows += chunk_rows
        usable_coordinate_rows += int(usable_mask.sum())
        nt_root_match_rows += int(matched_mask.sum())
        labelled_nt_rows += int(labelled_mask.sum())
        high_conf_nt_rows += int(high_conf_mask.sum())
        usable = merged.loc[usable_mask, ['neuropil', 'pre_root_id', 'pre_x_raw', 'pre_y_raw', 'pre_z_raw', 'distance_to_nearest_trachea_um']].copy()
        update_group_summaries(usable.assign(analysis_group='all_contacts'), ['analysis_group'], 'distance_to_nearest_trachea_um', overall_summary, overall_hist, histogram_edges)
        update_group_summaries(usable, ['neuropil'], 'distance_to_nearest_trachea_um', neuropil_summary, neuropil_hist, histogram_edges)
        high_nt = merged.loc[high_conf_usable, ['neuropil', 'pre_root_id', 'pre_x_raw', 'pre_y_raw', 'pre_z_raw', 'nt_type', 'nt_type_score', 'distance_to_nearest_trachea_um']].copy()
        update_group_summaries(high_nt, ['neuropil', 'nt_type'], 'distance_to_nearest_trachea_um', neuropil_nt_summary, neuropil_nt_hist, histogram_edges)
        if sample_per_neuropil > 0 and all_count_map and len(usable):
            expected_counts = usable['neuropil'].map(all_count_map)
            probabilities = np.array([sample_probability(int(value) if pd.notna(value) else None, sample_per_neuropil, sampling_overshoot) for value in expected_counts], dtype=np.float64)
            selected = rng.random(len(usable)) < probabilities
            if selected.any():
                all_sample_parts.append(usable.loc[selected].copy())
        if sample_per_neuropil_nt > 0 and nt_count_map and len(high_nt):
            keys = list(zip(high_nt['neuropil'].astype(str), high_nt['nt_type'].astype(str)))
            probabilities = np.array([sample_probability(nt_count_map.get(key), sample_per_neuropil_nt, sampling_overshoot) for key in keys], dtype=np.float64)
            selected = rng.random(len(high_nt)) < probabilities
            if selected.any():
                nt_sample_parts.append(high_nt.loc[selected].copy())
        elapsed = time.perf_counter() - started
        rate = total_rows / elapsed if elapsed > 0 else float('nan')
    elapsed = time.perf_counter() - started
    overall_df = summary_store_to_frame(overall_summary, ['analysis_group'])
    neuropil_df = summary_store_to_frame(neuropil_summary, ['neuropil'])
    if len(neuropil_df):
        neuropil_df = neuropil_df.sort_values(['mean_distance_um', 'neuropil'], ascending=[True, True], kind='mergesort').reset_index(drop=True)
    neuropil_nt_df = summary_store_to_frame(neuropil_nt_summary, ['neuropil', 'nt_type'])
    if len(neuropil_nt_df):
        neuropil_nt_df = neuropil_nt_df.sort_values(['neuropil', 'mean_distance_um', 'nt_type'], ascending=[True, True, True], kind='mergesort').reset_index(drop=True)
    overall_hist_df = histogram_store_to_frame(overall_hist, ['analysis_group'], histogram_edges)
    neuropil_hist_df = histogram_store_to_frame(neuropil_hist, ['neuropil'], histogram_edges)
    neuropil_nt_hist_df = histogram_store_to_frame(neuropil_nt_hist, ['neuropil', 'nt_type'], histogram_edges)
    if all_sample_parts:
        all_sample = pd.concat(all_sample_parts, ignore_index=True)
        all_sample = cap_group_samples(all_sample, ['neuropil'], sample_per_neuropil, random_seed + 1)
    else:
        all_sample = pd.DataFrame(columns=['neuropil', 'pre_root_id', 'pre_x_raw', 'pre_y_raw', 'pre_z_raw', 'distance_to_nearest_trachea_um'])
    if nt_sample_parts:
        nt_sample = pd.concat(nt_sample_parts, ignore_index=True)
        nt_sample = cap_group_samples(nt_sample, ['neuropil', 'nt_type'], sample_per_neuropil_nt, random_seed + 2)
    else:
        nt_sample = pd.DataFrame(columns=['neuropil', 'pre_root_id', 'pre_x_raw', 'pre_y_raw', 'pre_z_raw', 'nt_type', 'nt_type_score', 'distance_to_nearest_trachea_um'])
    paths = {'overall_summary': output_dir / 'overall_distance_summary.csv', 'neuropil_summary': output_dir / 'neuropil_distance_summary.csv', 'neuropil_nt_summary': output_dir / 'neuropil_nt_distance_summary.csv', 'overall_histogram': output_dir / 'overall_distance_histogram.csv', 'neuropil_histogram': output_dir / 'neuropil_distance_histograms.csv.gz', 'neuropil_nt_histogram': output_dir / 'neuropil_nt_distance_histograms.csv.gz', 'all_sample': output_dir / 'neuropil_violin_sample.csv.gz', 'nt_sample': output_dir / 'neuropil_nt_violin_sample.csv.gz'}
    overall_df.to_csv(paths['overall_summary'], index=False)
    neuropil_df.to_csv(paths['neuropil_summary'], index=False)
    neuropil_nt_df.to_csv(paths['neuropil_nt_summary'], index=False)
    overall_hist_df.to_csv(paths['overall_histogram'], index=False)
    neuropil_hist_df.to_csv(paths['neuropil_histogram'], index=False, compression='gzip')
    neuropil_nt_hist_df.to_csv(paths['neuropil_nt_histogram'], index=False, compression='gzip')
    all_sample.to_csv(paths['all_sample'], index=False, compression='gzip')
    nt_sample.to_csv(paths['nt_sample'], index=False, compression='gzip')
    run_summary = pd.DataFrame([{'metric': 'contact_rows_processed', 'value': total_rows, 'unit': 'rows'}, {'metric': 'usable_coordinate_rows', 'value': usable_coordinate_rows, 'unit': 'rows'}, {'metric': 'usable_coordinate_fraction', 'value': usable_coordinate_rows / total_rows if total_rows else float('nan'), 'unit': 'fraction'}, {'metric': 'nt_root_matched_rows', 'value': nt_root_match_rows, 'unit': 'rows'}, {'metric': 'nt_labelled_rows', 'value': labelled_nt_rows, 'unit': 'rows'}, {'metric': 'high_conf_nt_rows', 'value': high_conf_nt_rows, 'unit': 'rows'}, {'metric': 'high_conf_nt_fraction', 'value': high_conf_nt_rows / total_rows if total_rows else float('nan'), 'unit': 'fraction'}, {'metric': 'neuropil_groups', 'value': neuropil_df['neuropil'].nunique() if len(neuropil_df) else 0, 'unit': 'groups'}, {'metric': 'all_violin_sample_rows', 'value': len(all_sample), 'unit': 'rows'}, {'metric': 'nt_violin_sample_rows', 'value': len(nt_sample), 'unit': 'rows'}, {'metric': 'elapsed_seconds', 'value': elapsed, 'unit': 'seconds'}])
    run_summary_path = output_dir / 'run_summary.csv'
    run_summary.to_csv(run_summary_path, index=False)
    return 0

# Statistics
def build_superregion_mapping(observed_neuropils: list[str]):
    rows = [
        {"neuropil": neuropil, "super_region": super_region,
         "mapping_source": "Superregion_spreadsheet.ipynb"}
        for super_region in super_order
        for neuropil in superregions[super_region]
    ]
    mapping = pd.DataFrame(rows)
    if mapping.duplicated("neuropil").any():
        duplicated = sorted(mapping.loc[mapping.duplicated("neuropil", keep=False), "neuropil"].unique())
        raise RuntimeError("Duplicate neuropils in superregion mapping: " + ", ".join(duplicated))

    observed = {str(x) for x in observed_neuropils}
    mapped = set(mapping["neuropil"])
    observed_unmapped = sorted(observed - mapped)
    if observed_unmapped:
        raise RuntimeError(
            "Observed neuropils are absent from the super-region mapping: "
            + ", ".join(observed_unmapped)
        )
    mapping_observed = mapping[mapping["neuropil"].isin(observed)].copy()
    mapped_not_observed = sorted(mapped - observed)
    return mapping_observed, observed_unmapped, mapped_not_observed

def safe_filename(value: object) -> str:
    text = re.sub('[^A-Za-z0-9._-]+', '_', str(value).strip()).strip('_')
    return text or 'unnamed'

def holm_adjust(p_values: Iterable[float]) -> np.ndarray:
    values = np.asarray(list(p_values), dtype=np.float64)
    adjusted = np.full(len(values), np.nan, dtype=np.float64)
    finite_indices = np.flatnonzero(np.isfinite(values))
    if len(finite_indices) == 0:
        return adjusted
    ordered = finite_indices[np.argsort(values[finite_indices])]
    number = len(ordered)
    running_maximum = 0.0
    for rank, original_index in enumerate(ordered):
        candidate = min(1.0, values[original_index] * (number - rank))
        running_maximum = max(running_maximum, candidate)
        adjusted[original_index] = running_maximum
    return adjusted

def rank_biserial_from_u(u_statistic: float, n_a: int, n_b: int) -> float:
    if n_a <= 0 or n_b <= 0:
        return float('nan')
    return 2.0 * float(u_statistic) / float(n_a * n_b) - 1.0

def load_inputs(input_dir: Path, include_missing: bool) -> dict[str, pd.DataFrame]:
    files = {'all_exact': require_file(input_dir / 'neuropil_distance_summary.csv'), 'nt_exact': require_file(input_dir / 'neuropil_nt_distance_summary.csv'), 'all_sample': require_file(input_dir / 'neuropil_violin_sample.csv.gz'), 'nt_sample': require_file(input_dir / 'neuropil_nt_violin_sample.csv.gz')}
    tables = {key: pd.read_csv(path) for key, path in files.items()}
    for key, table in tables.items():
        if 'neuropil' not in table.columns:
            raise KeyError(f'{key} has no `neuropil` column.')
        table['neuropil'] = clean_text(table['neuropil']).fillna('<MISSING>').astype(str)
        if not include_missing:
            table.drop(table[table['neuropil'] == '<MISSING>'].index, inplace=True)
        table = table.loc[~table['neuropil'].astype(str).str.upper().isin(excluded_regions)].copy()
        tables[key] = table
    for key in ['nt_exact', 'nt_sample']:
        table = tables[key]
        if 'nt_type' not in table.columns:
            raise KeyError(f'{key} has no `nt_type` column.')
        table['nt_type'] = clean_text(table['nt_type']).fillna('<MISSING>').astype(str).str.upper()
    for key in ['all_sample', 'nt_sample']:
        table = tables[key]
        if dist_col not in table.columns:
            raise KeyError(f'{key} has no `{dist_col}` column.')
        table[dist_col] = pd.to_numeric(table[dist_col], errors='coerce')
        tables[key] = table.dropna(subset=[dist_col]).copy()
    for key in ['all_exact', 'nt_exact']:
        if 'n' not in tables[key].columns:
            raise KeyError(f'{key} has no exact `n` column.')
        tables[key]['n'] = pd.to_numeric(tables[key]['n'], errors='coerce')
    return tables

def descriptive_summary(sample: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    result = sample.groupby(group_columns, dropna=False)[dist_col].agg(sample_n='size', sample_mean_um='mean', sample_sd_um='std', sample_median_um='median', sample_q1_um=lambda values: values.quantile(0.25), sample_q3_um=lambda values: values.quantile(0.75), sample_min_um='min', sample_max_um='max').reset_index()
    result['sample_iqr_um'] = result['sample_q3_um'] - result['sample_q1_um']
    return result

def combine_exact_and_sample(exact: pd.DataFrame, sample: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    exact_columns = group_columns + [column for column in ['n', 'mean_distance_um', 'standard_deviation_um', 'minimum_distance_um', 'maximum_distance_um'] if column in exact.columns]
    exact_part = exact[exact_columns].copy()
    descriptive = descriptive_summary(sample, group_columns)
    result = exact_part.merge(descriptive, on=group_columns, how='outer', validate='one_to_one')
    return result

def add_contact_weights(sample: pd.DataFrame, exact: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    sample_counts = sample.groupby(group_columns, as_index=False).size().rename(columns={'size': 'available_sample_n'})
    exact_counts = exact[group_columns + ['n']].rename(columns={'n': 'exact_n'})
    weights = exact_counts.merge(sample_counts, on=group_columns, how='inner', validate='one_to_one')
    weights['row_weight'] = weights['exact_n'] / weights['available_sample_n']
    result = sample.merge(weights[group_columns + ['exact_n', 'available_sample_n', 'row_weight']], on=group_columns, how='inner', validate='many_to_one')
    return result

def contact_weighted_resample(weighted_sample: pd.DataFrame, stratum_columns: list[str], target_n: int, random_seed: int) -> pd.DataFrame:
    if target_n <= 0:
        raise ValueError('Target weighted sample n must be positive.')
    rng = np.random.default_rng(random_seed)
    parts = []
    for _, group in weighted_sample.groupby(stratum_columns, dropna=False, sort=True):
        weights = pd.to_numeric(group['row_weight'], errors='coerce').to_numpy(dtype=np.float64)
        valid = np.isfinite(weights) & (weights > 0)
        group = group.loc[valid].copy()
        weights = weights[valid]
        if len(group) == 0:
            continue
        probabilities = weights / weights.sum()
        indices = rng.choice(len(group), size=target_n, replace=True, p=probabilities)
        sampled = group.iloc[indices].copy()
        parts.append(sampled)
    if not parts:
        return pd.DataFrame(columns=weighted_sample.columns)
    return pd.concat(parts, ignore_index=True)

def aggregate_exact_n(exact: pd.DataFrame, mapping: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    merged = exact.merge(mapping, on='neuropil', how='inner', validate='many_to_one')
    return merged.groupby(group_columns, as_index=False)['n'].sum()

def kruskal_test(sample: pd.DataFrame, group_column: str, family: str, level: str, nt_type: str | None, minimum_n: int) -> dict:
    arrays = []
    labels = []
    group_ns = []
    for group_name, subset in sample.groupby(group_column, sort=True):
        values = subset[dist_col].dropna().to_numpy(dtype=np.float64)
        if len(values) >= minimum_n:
            arrays.append(values)
            labels.append(str(group_name))
            group_ns.append(int(len(values)))
    row = {'analysis_family': family, 'anatomical_level': level, 'nt_type': nt_type, 'group_column': group_column, 'number_of_groups': int(len(arrays)), 'sample_n_total': int(sum(group_ns)), 'minimum_group_sample_n': min(group_ns) if group_ns else np.nan, 'maximum_group_sample_n': max(group_ns) if group_ns else np.nan, 'kruskal_h': np.nan, 'degrees_of_freedom': len(arrays) - 1, 'p_value': np.nan, 'status': ''}
    if len(arrays) < 2:
        row['status'] = 'fewer_than_two_eligible_groups'
        return row
    result = kruskal(*arrays, nan_policy='omit')
    row['kruskal_h'] = float(result.statistic)
    row['p_value'] = float(result.pvalue)
    row['status'] = 'success'
    return row

def pairwise_tests(sample: pd.DataFrame, exact: pd.DataFrame, group_column: str, family: str, level: str, nt_type: str | None, minimum_n: int) -> pd.DataFrame:
    groups: dict[str, np.ndarray] = {}
    for group_name, subset in sample.groupby(group_column, sort=True):
        values = subset[dist_col].dropna().to_numpy(dtype=np.float64)
        if len(values) >= minimum_n:
            groups[str(group_name)] = values
    exact_lookup = {str(row[group_column]): int(row['n']) for _, row in exact.iterrows() if pd.notna(row.get(group_column)) and pd.notna(row.get('n'))}
    rows = []
    for group_a, group_b in itertools.combinations(sorted(groups.keys()), 2):
        values_a = groups[group_a]
        values_b = groups[group_b]
        result = mannwhitneyu(values_a, values_b, alternative='two-sided', method='asymptotic')
        median_a = float(np.median(values_a))
        median_b = float(np.median(values_b))
        rows.append({'analysis_family': family, 'anatomical_level': level, 'nt_type': nt_type, 'group_a': group_a, 'group_b': group_b, 'exact_n_a': exact_lookup.get(group_a, np.nan), 'exact_n_b': exact_lookup.get(group_b, np.nan), 'sample_n_a': int(len(values_a)), 'sample_n_b': int(len(values_b)), 'median_a_um': median_a, 'median_b_um': median_b, 'median_difference_a_minus_b_um': median_a - median_b, 'absolute_median_difference_um': abs(median_a - median_b), 'mannwhitney_u': float(result.statistic), 'p_value': float(result.pvalue), 'rank_biserial_a_vs_b': rank_biserial_from_u(result.statistic, len(values_a), len(values_b))})
    if not rows:
        return pd.DataFrame()
    table = pd.DataFrame(rows)
    table['p_holm_within_family'] = holm_adjust(table['p_value'])
    table['significant_after_holm_0_05'] = table['p_holm_within_family'] < 0.05
    table['absolute_rank_biserial'] = table['rank_biserial_a_vs_b'].abs()
    return table.sort_values(['p_holm_within_family', 'absolute_rank_biserial'], ascending=[True, False], kind='mergesort').reset_index(drop=True)

def run_all_statistics(all_subregion_sample: pd.DataFrame, all_subregion_exact: pd.DataFrame, all_super_sample: pd.DataFrame, all_super_exact: pd.DataFrame, nt_subregion_sample: pd.DataFrame, nt_subregion_exact: pd.DataFrame, nt_super_sample: pd.DataFrame, nt_super_exact: pd.DataFrame, minimum_n: int, skip_pairwise: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    omnibus_rows = []
    pairwise_parts = []
    families = [{'sample': all_subregion_sample, 'exact': all_subregion_exact, 'group': 'neuropil', 'family': 'all_contacts_across_subregions', 'level': 'subregion', 'nt_type': None}, {'sample': all_super_sample, 'exact': all_super_exact, 'group': 'super_region', 'family': 'all_contacts_across_superregions', 'level': 'super_region', 'nt_type': None}]
    present_nt = [nt_type for nt_type in nt_order if nt_type in set(nt_subregion_sample['nt_type'])]
    extras = sorted(set(nt_subregion_sample['nt_type']) - set(present_nt))
    for nt_type in present_nt + extras:
        sub_sample = nt_subregion_sample[nt_subregion_sample['nt_type'] == nt_type].copy()
        sub_exact = nt_subregion_exact[nt_subregion_exact['nt_type'] == nt_type].copy()
        super_sample = nt_super_sample[nt_super_sample['nt_type'] == nt_type].copy()
        super_exact = nt_super_exact[nt_super_exact['nt_type'] == nt_type].copy()
        families.extend([{'sample': sub_sample, 'exact': sub_exact, 'group': 'neuropil', 'family': f'{nt_type}_across_subregions', 'level': 'subregion', 'nt_type': nt_type}, {'sample': super_sample, 'exact': super_exact, 'group': 'super_region', 'family': f'{nt_type}_across_superregions', 'level': 'super_region', 'nt_type': nt_type}])
    for specification in families:
        sample = specification['sample']
        exact = specification['exact']
        group_column = specification['group']
        family = specification['family']
        level = specification['level']
        nt_type = specification['nt_type']
        omnibus_rows.append(kruskal_test(sample=sample, group_column=group_column, family=family, level=level, nt_type=nt_type, minimum_n=minimum_n))
        if not skip_pairwise:
            pairwise = pairwise_tests(sample=sample, exact=exact, group_column=group_column, family=family, level=level, nt_type=nt_type, minimum_n=minimum_n)
            if len(pairwise):
                pairwise_parts.append(pairwise)
    omnibus = pd.DataFrame(omnibus_rows)
    omnibus['p_holm_across_omnibus_families'] = holm_adjust(omnibus['p_value'])
    omnibus['significant_after_omnibus_holm_0_05'] = omnibus['p_holm_across_omnibus_families'] < 0.05
    pairwise = pd.concat(pairwise_parts, ignore_index=True) if pairwise_parts else pd.DataFrame()
    return (omnibus, pairwise)

def summarize_results(input_dir, output_dir, groups_per_page, minimum_sample_n, superregion_sample_n, superregion_nt_sample_n, plot_upper_quantile, top_effects_per_family, random_seed) -> int:
    input_dir = input_dir.expanduser().resolve()
    output_dir = output_dir.expanduser().resolve()
    if groups_per_page <= 0:
        raise ValueError('--groups-per-page must be positive.')
    if not 0.5 < plot_upper_quantile <= 1.0:
        raise ValueError('--plot-upper-quantile must be >0.5 and <=1.')
    output_dir.mkdir(parents=True, exist_ok=True)
    figure_dir = output_dir / 'FINAL_FIGURES'
    table_dir = output_dir / 'FINAL_TABLES'
    figure_dir.mkdir(parents=True, exist_ok=True)
    table_dir.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    tables = load_inputs(input_dir, include_missing=False)
    all_exact = tables['all_exact']
    nt_exact = tables['nt_exact']
    all_sample = tables['all_sample']
    nt_sample = tables['nt_sample']
    observed_neuropils = sorted(set(all_exact['neuropil'].astype(str)) | set(all_sample['neuropil'].astype(str)))
    mapping, observed_unmapped_superregions, mapped_not_observed = build_superregion_mapping(observed_neuropils)
    mapping_path = table_dir / '00_superregion_mapping.csv'
    mapping.to_csv(mapping_path, index=False)
    unmapped_path = table_dir / '00_subregions_not_in_superregion_mapping.csv'
    pd.DataFrame({'neuropil': observed_unmapped_superregions, 'status': 'excluded_from_superregion_analysis', 'reason': "Absent from the Superregion_spreadsheet.ipynb mapping"}).to_csv(unmapped_path, index=False)
    all_subregion_summary = combine_exact_and_sample(all_exact, all_sample, ['neuropil'])
    nt_subregion_summary = combine_exact_and_sample(nt_exact, nt_sample, ['neuropil', 'nt_type'])
    all_weighted = add_contact_weights(all_sample, all_exact, ['neuropil']).merge(mapping, on='neuropil', how='inner', validate='many_to_one')
    nt_weighted = add_contact_weights(nt_sample, nt_exact, ['neuropil', 'nt_type']).merge(mapping, on='neuropil', how='inner', validate='many_to_one')
    all_super_sample = contact_weighted_resample(all_weighted, ['super_region'], superregion_sample_n, random_seed + 1)
    nt_super_sample = contact_weighted_resample(nt_weighted, ['super_region', 'nt_type'], superregion_nt_sample_n, random_seed + 2)
    all_super_exact = aggregate_exact_n(all_exact, mapping, ['super_region'])
    nt_super_exact = aggregate_exact_n(nt_exact, mapping, ['super_region', 'nt_type'])
    all_super_summary = combine_exact_and_sample(all_super_exact, all_super_sample, ['super_region'])
    nt_super_summary = combine_exact_and_sample(nt_super_exact, nt_super_sample, ['super_region', 'nt_type'])
    descriptive_paths = {'all_subregion': table_dir / '01_all_contacts_subregion_exact_n_descriptives.csv', 'all_superregion': table_dir / '02_all_contacts_superregion_exact_n_descriptives.csv', 'nt_subregion': table_dir / '03_nt_subregion_exact_n_descriptives.csv', 'nt_superregion': table_dir / '04_nt_superregion_exact_n_descriptives.csv'}
    all_subregion_summary.to_csv(descriptive_paths['all_subregion'], index=False)
    all_super_summary.to_csv(descriptive_paths['all_superregion'], index=False)
    nt_subregion_summary.to_csv(descriptive_paths['nt_subregion'], index=False)
    nt_super_summary.to_csv(descriptive_paths['nt_superregion'], index=False)
    figure_paths: list[Path] = []
    all_subregion_order = sorted(all_subregion_summary.dropna(subset=['neuropil', 'sample_median_um'])['neuropil'].astype(str).tolist())
    all_subregion_violin_pdf = figure_dir / '01_all_contacts_subregions_violin_multipage.pdf'
    write_multipage_violin_pdf(sample=all_sample, exact=all_exact, group_column='neuropil', ordered_groups=all_subregion_order, title_prefix='All synaptic contacts across subregions', output_pdf=all_subregion_violin_pdf, groups_per_page=groups_per_page, upper_quantile=plot_upper_quantile)
    figure_paths.append(all_subregion_violin_pdf)
    subregion_ranked_figure = ranked_median_iqr_figure(all_subregion_summary, 'neuropil', 'Median synapse-to-trachea distance across subregions')
    figure_paths.extend(save_single_figure(subregion_ranked_figure, figure_dir / '02_all_contacts_subregions_median_iqr'))
    all_super_order = sorted(all_super_summary.dropna(subset=['super_region', 'sample_median_um'])['super_region'].astype(str).tolist())
    superregion_violin_figure = violin_page(sample=all_super_sample, exact=all_super_exact, group_column='super_region', groups=all_super_order, title='All synaptic contacts across super-regions', upper_quantile=plot_upper_quantile)
    if superregion_violin_figure is not None:
        figure_paths.extend(save_single_figure(superregion_violin_figure, figure_dir / '03_all_contacts_superregions_violin'))
    superregion_ranked_figure = ranked_median_iqr_figure(all_super_summary, 'super_region', 'Median synapse-to-trachea distance across super-regions')
    figure_paths.extend(save_single_figure(superregion_ranked_figure, figure_dir / '04_all_contacts_superregions_median_iqr'))
    present_nt = [nt_type for nt_type in nt_order if nt_type in set(nt_sample['nt_type'])]
    for nt_type in present_nt:
        nt_sample_subset = nt_sample[nt_sample['nt_type'] == nt_type].copy()
        nt_exact_subset = nt_exact[nt_exact['nt_type'] == nt_type].copy()
        nt_summary_subset = nt_subregion_summary[nt_subregion_summary['nt_type'] == nt_type].copy()
        order = sorted(nt_summary_subset.dropna(subset=['neuropil', 'sample_median_um'])['neuropil'].astype(str).tolist())
        output_pdf = figure_dir / ('05_nt_subregions_' + safe_filename(nt_type) + '_violin_multipage.pdf')
        write_multipage_violin_pdf(sample=nt_sample_subset, exact=nt_exact_subset, group_column='neuropil', ordered_groups=order, title_prefix=f'{nt_type} synaptic contacts across subregions', output_pdf=output_pdf, groups_per_page=groups_per_page, upper_quantile=plot_upper_quantile)
        figure_paths.append(output_pdf)
    six_panel_figure = six_panel_superregion_nt_figure(nt_super_sample, nt_super_exact, plot_upper_quantile)
    figure_paths.extend(save_single_figure(six_panel_figure, figure_dir / '06_nt_superregions_six_panels'))
    subregion_heatmap = heatmap_figure(nt_subregion_summary, row_column='neuropil', title='Median distance by subregion and neurotransmitter', annotate=False)
    figure_paths.extend(save_single_figure(subregion_heatmap, figure_dir / '07_nt_subregions_median_heatmap'))
    superregion_heatmap = heatmap_figure(nt_super_summary, row_column='super_region', title='Median distance by super-region and neurotransmitter', annotate=True)
    figure_paths.extend(save_single_figure(superregion_heatmap, figure_dir / '08_nt_superregions_median_heatmap'))
    omnibus, pairwise = run_all_statistics(all_subregion_sample=all_sample, all_subregion_exact=all_exact, all_super_sample=all_super_sample, all_super_exact=all_super_exact, nt_subregion_sample=nt_sample, nt_subregion_exact=nt_exact, nt_super_sample=nt_super_sample, nt_super_exact=nt_super_exact, minimum_n=minimum_sample_n, skip_pairwise=False)
    omnibus_path = table_dir / '05_kruskal_wallis_summary.csv'
    omnibus.to_csv(omnibus_path, index=False)
    pairwise_path = table_dir / '06_pairwise_mannwhitney_holm_all.csv.gz'
    pairwise.to_csv(pairwise_path, index=False, compression='gzip')
    if len(pairwise):
        top_effects = pairwise.sort_values(['analysis_family', 'absolute_rank_biserial', 'absolute_median_difference_um'], ascending=[True, False, False], kind='mergesort').groupby('analysis_family', group_keys=False).head(top_effects_per_family).reset_index(drop=True)
    else:
        top_effects = pd.DataFrame()
    top_effects_path = table_dir / '07_pairwise_top_effects.csv'
    top_effects.to_csv(top_effects_path, index=False)
    effect_summary_columns = [column for column in ['analysis_family', 'anatomical_level', 'nt_type', 'group_a', 'group_b', 'exact_n_a', 'exact_n_b', 'sample_n_a', 'sample_n_b', 'median_a_um', 'median_b_um', 'median_difference_a_minus_b_um', 'rank_biserial_a_vs_b', 'absolute_rank_biserial', 'p_holm_within_family', 'significant_after_holm_0_05'] if column in top_effects.columns]
    exact_n_effect_path = table_dir / '08_exact_n_and_effect_size_summary.csv'
    top_effects[effect_summary_columns].to_csv(exact_n_effect_path, index=False)
    table_paths = [mapping_path, unmapped_path] + list(descriptive_paths.values()) + [omnibus_path, pairwise_path, top_effects_path, exact_n_effect_path]
    metrics = {'exact_contact_rows_subregions': int(all_exact['n'].fillna(0).sum()), 'exact_contact_rows_superregions': int(all_super_exact['n'].fillna(0).sum()), 'subregion_count': int(all_exact['neuropil'].nunique()), 'mapped_subregion_count': int(mapping['neuropil'].nunique()), 'subregions_unmapped': observed_unmapped_superregions, 'mapped_neuropils_absent_from_data': mapped_not_observed, 'superregion_count': int(all_super_exact['super_region'].nunique()), 'nt_types': present_nt, 'omnibus_test_count': int(len(omnibus)), 'pairwise_test_count': int(len(pairwise)), 'figure_file_count': int(len(figure_paths)), 'table_file_count': int(len(table_paths)), 'elapsed_seconds': float(time.perf_counter() - started), 'superregion_mapping_source': 'Superregion_spreadsheet.ipynb', 'embedded_superregion_mapping': superregions, 'input_dir': str(input_dir), 'output_dir': str(output_dir)}
    summary_json = output_dir / 'analysis_summary.json'
    summary_json.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    return 0

# Figures 01-08
all_color = "#3F6F8F"

nt_colors = {
    "ACH": "#2020df",
    "DA": "#963939",
    "GABA": "#df2020",
    "GLUT": "#df9c20",
    "OCT": "#107010",
    "SER": "#701070",
}
nt_other_color = "#7A7A7A"

median_color = "#202020"
grid_color = "#D9D9D9"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 13,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 13,
    "axes.linewidth": 0.65,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})

def _style_axis(axis: plt.Axes, grid_x: bool = True) -> None:
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.spines["left"].set_linewidth(0.65)
    axis.spines["bottom"].set_linewidth(0.65)
    axis.tick_params(direction="out", width=0.6, length=3, pad=2)
    axis.set_axisbelow(True)
    if grid_x:
        axis.grid(axis="x", color=grid_color, linewidth=0.45, alpha=0.7)

def _style_violin_parts(violin: dict, face_color: str) -> None:
    for body in violin.get("bodies", []):
        body.set_facecolor(face_color)
        body.set_edgecolor(face_color)
        body.set_alpha(0.58)
        body.set_linewidth(0.55)

    for key in ["cmedians", "cbars", "cmins", "cmaxes"]:
        artist = violin.get(key)
        if artist is not None:
            artist.set_color(median_color)
            artist.set_linewidth(0.8)

def figure_xmax(arrays: list[np.ndarray], upper_quantile: float) -> float | None:
    finite = [a[np.isfinite(a)] for a in arrays if len(a)]
    if not finite:
        return None
    combined = np.concatenate(finite)
    if len(combined) == 0:
        return None
    return float(np.quantile(combined, upper_quantile))

def _nt_color_from_sample(sample: pd.DataFrame) -> str:
    if "nt_type" not in sample.columns:
        return all_color
    nts = sample["nt_type"].dropna().astype(str).unique().tolist()
    if len(nts) == 1:
        return nt_colors.get(nts[0], nt_other_color)
    return all_color

def save_single_figure(figure: plt.Figure, base_path: Path) -> list[Path]:
    path = Path(base_path).with_suffix(".svg")
    figure.savefig(path, format="svg", bbox_inches="tight", pad_inches=0.04)
    plt.close(figure)
    return [path]

def violin_page(
    sample: pd.DataFrame,
    exact: pd.DataFrame | None = None,
    group_column: str | None = None,
    groups: list[str] | None = None,
    title: str = "",
    upper_quantile: float = 0.995,
    color: str | None = None,
) -> plt.Figure | None:
    del exact

    if group_column is None or groups is None:
        raise ValueError("group_column and groups are required.")

    if color is None:
        color = _nt_color_from_sample(sample)

    arrays = []
    labels = []
    medians = []

    for group_name in groups:
        values = (
            sample.loc[
                sample[group_column].astype(str) == str(group_name),
                dist_col,
            ]
            .dropna()
            .to_numpy(dtype=float)
        )
        if len(values) == 0:
            continue
        arrays.append(values)
        labels.append(str(group_name))
        medians.append(float(np.median(values)))

    if not arrays:
        return None

    height = max(3.2, 0.34 * len(arrays) + 1.15)
    figure, axis = plt.subplots(figsize=(7.0, height))
    positions = np.arange(1, len(arrays) + 1)

    violin = axis.violinplot(
        arrays,
        positions=positions,
        vert=False,
        widths=0.78,
        showmeans=False,
        showmedians=False,
        showextrema=False,
        points=100,
        bw_method="scott",
    )
    _style_violin_parts(violin, color)

    axis.scatter(
        medians,
        positions,
        s=32,
        facecolor=median_color,
        edgecolor="white",
        linewidth=0.35,
        zorder=4,
        label="Median",
    )

    axis.set_yticks(positions)
    axis.set_yticklabels(labels)
    axis.invert_yaxis()

    xmax = figure_xmax(arrays, upper_quantile)
    if xmax is not None and np.isfinite(xmax) and xmax > 0:
        axis.set_xlim(0, xmax)

    axis.set_xlabel("Distance to nearest tracheole point (µm)")
    axis.set_title(title, loc="left", fontweight="semibold", pad=7)
    _style_axis(axis, grid_x=True)
    axis.legend(
        frameon=False,
        loc="lower right",
        fontsize=15,
        markerscale=1.65,
        handletextpad=0.7,
        borderaxespad=0.65,
        labelspacing=0.55,
    )
    figure.tight_layout(pad=0.6)
    return figure

def write_multipage_violin_pdf(
    sample: pd.DataFrame,
    exact: pd.DataFrame | None = None,
    group_column: str | None = None,
    ordered_groups: list[str] | None = None,
    title_prefix: str = "",
    output_pdf: Path | None = None,
    groups_per_page: int = 20,
    upper_quantile: float = 0.995,
    color: str | None = None,
) -> None:
    del exact
    if group_column is None or ordered_groups is None or output_pdf is None:
        raise ValueError("group_column, ordered_groups, and output_pdf are required.")

    if color is None:
        color = _nt_color_from_sample(sample)

    output_pdf = Path(output_pdf).with_suffix(".pdf")
    with PdfPages(output_pdf) as pdf:
        for start in range(0, len(ordered_groups), groups_per_page):
            page_groups = ordered_groups[start:start + groups_per_page]
            figure = violin_page(
                sample=sample,
                group_column=group_column,
                groups=page_groups,
                title=title_prefix,
                upper_quantile=upper_quantile,
                color=color,
            )
            if figure is not None:
                pdf.savefig(figure, bbox_inches="tight", pad_inches=0.04)
                plt.close(figure)

def ranked_median_iqr_figure(
    summary: pd.DataFrame,
    group_column: str,
    title: str,
) -> plt.Figure:
    table = (
        summary.dropna(
            subset=[group_column, "sample_median_um", "sample_q1_um", "sample_q3_um"]
        )
        .assign(_group_sort=lambda df: df[group_column].astype(str))
        .sort_values("_group_sort", ascending=True, kind="mergesort")
        .drop(columns="_group_sort")
        .reset_index(drop=True)
    )

    height = max(3.2, 0.27 * len(table) + 1.1)
    figure, axis = plt.subplots(figsize=(6.8, height))
    positions = np.arange(len(table))

    lower = table["sample_median_um"] - table["sample_q1_um"]
    upper = table["sample_q3_um"] - table["sample_median_um"]

    axis.errorbar(
        table["sample_median_um"],
        positions,
        xerr=np.vstack([lower, upper]),
        fmt="o",
        color=all_color,
        ecolor=all_color,
        markerfacecolor=all_color,
        markeredgecolor="white",
        markeredgewidth=0.35,
        markersize=4.0,
        capsize=2.0,
        capthick=0.7,
        elinewidth=0.8,
    )

    axis.set_yticks(positions)
    axis.set_yticklabels(table[group_column].astype(str))
    axis.invert_yaxis()
    axis.set_xlabel("Median distance to nearest tracheole (µm)")
    axis.set_title(title, loc="left", fontweight="semibold", pad=7)
    _style_axis(axis, grid_x=True)
    figure.tight_layout(pad=0.6)
    return figure

def six_panel_superregion_nt_figure(
    sample: pd.DataFrame,
    exact: pd.DataFrame | None = None,
    upper_quantile: float = 0.995,
) -> plt.Figure:
    del exact

    nt_types = [nt for nt in nt_order if nt in set(sample["nt_type"].astype(str))]
    figure, axes = plt.subplots(nrows=3, ncols=2, figsize=(10.2, 11.2))
    axes_flat = axes.ravel()

    for axis_index, axis in enumerate(axes_flat):
        if axis_index >= len(nt_types):
            axis.axis("off")
            continue

        nt_type = nt_types[axis_index]
        subset = sample.loc[sample["nt_type"] == nt_type].copy()
        groups = sorted(subset["super_region"].dropna().astype(str).unique().tolist())

        arrays = []
        labels = []
        for group_name in groups:
            values = (
                subset.loc[
                    subset["super_region"].astype(str) == group_name,
                    dist_col,
                ]
                .dropna()
                .to_numpy(dtype=float)
            )
            if len(values):
                arrays.append(values)
                labels.append(group_name)

        positions = np.arange(1, len(arrays) + 1)
        color = nt_colors.get(nt_type, nt_other_color)

        violin = axis.violinplot(
            arrays,
            positions=positions,
            vert=False,
            widths=0.76,
            showmeans=False,
            showmedians=True,
            showextrema=False,
            points=100,
            bw_method="scott",
        )
        _style_violin_parts(violin, color)

        axis.set_yticks(positions)
        axis.set_yticklabels(labels, fontsize=11)
        axis.invert_yaxis()

        xmax = figure_xmax(arrays, upper_quantile)
        if xmax is not None and np.isfinite(xmax) and xmax > 0:
            axis.set_xlim(0, xmax)

        axis.set_title(nt_type, loc="left", fontweight="semibold", pad=5)
        axis.set_xlabel("Distance (µm)")
        _style_axis(axis, grid_x=True)

    figure.suptitle(
        "Neurotransmitter-specific synapse-to-trachea distances across super-regions",
        fontsize=14,
        fontweight="semibold",
        x=0.08,
        ha="left",
        y=0.995,
    )
    figure.tight_layout(rect=(0, 0, 1, 0.975), h_pad=1.0, w_pad=1.0)
    return figure

def heatmap_figure(
    summary: pd.DataFrame,
    row_column: str,
    title: str,
    annotate: bool,
) -> plt.Figure:
    pivot = summary.pivot_table(
        index=row_column,
        columns="nt_type",
        values="sample_median_um",
        aggfunc="first",
    )

    ordered_columns = [nt for nt in nt_order if nt in pivot.columns]
    extra_columns = sorted([c for c in pivot.columns if c not in ordered_columns])
    pivot = pivot[ordered_columns + extra_columns]
    pivot = pivot.loc[sorted(pivot.index.astype(str).tolist())]

    height = max(4.2, 0.22 * len(pivot) + 1.45)
    figure, axis = plt.subplots(figsize=(6.7, height))

    matrix = pivot.to_numpy(dtype=float)
    image = axis.imshow(matrix, aspect="auto", cmap="viridis", interpolation="nearest")

    axis.set_xticks(np.arange(len(pivot.columns)))
    axis.set_xticklabels(pivot.columns)
    axis.set_yticks(np.arange(len(pivot.index)))
    axis.set_yticklabels(pivot.index, fontsize=9.5 if len(pivot) > 30 else 11)
    axis.set_xlabel("Predicted neurotransmitter")
    axis.set_ylabel("")
    axis.set_title(title, loc="left", fontweight="semibold", pad=7)
    axis.tick_params(length=0)

    for spine in axis.spines.values():
        spine.set_visible(False)

    color_bar = figure.colorbar(image, ax=axis, fraction=0.035, pad=0.025)
    color_bar.outline.set_linewidth(0.5)
    color_bar.ax.tick_params(width=0.5, length=2)
    color_bar.set_label("Median distance to nearest tracheole (µm)")

    if annotate:
        finite = matrix[np.isfinite(matrix)]
        midpoint = float(np.nanmedian(finite)) if len(finite) else 0.0
        for row_index in range(len(pivot.index)):
            for column_index in range(len(pivot.columns)):
                value = matrix[row_index, column_index]
                if np.isfinite(value):
                    text_color = "white" if value <= midpoint else "black"
                    axis.text(
                        column_index,
                        row_index,
                        f"{value:.1f}",
                        ha="center",
                        va="center",
                        fontsize=10.5,
                        color=text_color,
                    )

    figure.tight_layout(pad=0.6)
    return figure

# Effect-size figures
def validate_and_prepare(dataframe: pd.DataFrame) -> pd.DataFrame:
    required = {'analysis_family', 'anatomical_level', 'nt_type', 'group_a', 'group_b', 'exact_n_a', 'exact_n_b', 'sample_n_a', 'sample_n_b', 'median_a_um', 'median_b_um', 'median_difference_a_minus_b_um', 'absolute_median_difference_um', 'rank_biserial_a_vs_b', 'absolute_rank_biserial', 'p_holm_within_family', 'significant_after_holm_0_05'}
    missing = required - set(dataframe.columns)
    if missing:
        raise KeyError('Pairwise table is missing required columns:\n' + '\n'.join(sorted(missing)))
    result = dataframe.copy()
    numeric_columns = ['exact_n_a', 'exact_n_b', 'sample_n_a', 'sample_n_b', 'median_a_um', 'median_b_um', 'median_difference_a_minus_b_um', 'absolute_median_difference_um', 'rank_biserial_a_vs_b', 'absolute_rank_biserial', 'p_holm_within_family']
    for column in numeric_columns:
        result[column] = pd.to_numeric(result[column], errors='coerce')
    result['significant_after_holm_0_05'] = result['significant_after_holm_0_05'].astype(str).str.strip().str.lower().isin(['true', '1', 'yes', 'y'])
    result['nt_label'] = result['nt_type'].astype('string').fillna('All contacts')
    result['comparison'] = result['group_a'].astype(str) + ' vs ' + result['group_b'].astype(str)
    result['direction_text'] = np.where(result['median_difference_a_minus_b_um'] >= 0, result['group_a'].astype(str) + ' farther', result['group_b'].astype(str) + ' farther')
    result['effect_magnitude'] = pd.cut(result['absolute_rank_biserial'], bins=[-np.inf, 0.1, 0.3, 0.5, np.inf], labels=['negligible (<0.10)', 'small (0.10–0.29)', 'moderate (0.30–0.49)', 'large (≥0.50)'], right=False)
    result = result.dropna(subset=['absolute_median_difference_um', 'absolute_rank_biserial', 'median_difference_a_minus_b_um', 'rank_biserial_a_vs_b']).copy()
    return result

def save_figure(figure: plt.Figure, output_stem: Path) -> list[Path]:
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    paths = [output_stem.with_suffix('.pdf'), output_stem.with_suffix('.png'), output_stem.with_suffix('.svg')]
    figure.savefig(paths[0], bbox_inches='tight')
    figure.savefig(paths[1], dpi=300, bbox_inches='tight')
    figure.savefig(paths[2], bbox_inches='tight')
    plt.close(figure)
    return paths

def hide_top_right_spines(axis: plt.Axes) -> None:
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)

def plot_effect_vs_difference(dataframe: pd.DataFrame, title: str, output_stem: Path, label_n: int) -> list[Path]:
    figure, axis = plt.subplots(figsize=(7.2, 5.8))
    significant = dataframe['significant_after_holm_0_05'].to_numpy(dtype=bool)
    axis.scatter(dataframe.loc[~significant, 'absolute_median_difference_um'], dataframe.loc[~significant, 'absolute_rank_biserial'], s=16, alpha=0.35, label='Not Holm-significant')
    axis.scatter(dataframe.loc[significant, 'absolute_median_difference_um'], dataframe.loc[significant, 'absolute_rank_biserial'], s=18, alpha=0.55, label='Holm-significant')
    axis.axhline(0.1, linewidth=0.8, linestyle='--')
    axis.axhline(0.3, linewidth=0.8, linestyle='--')
    axis.axhline(0.5, linewidth=0.8, linestyle='--')
    axis.text(0.99, 0.105, 'small', transform=axis.get_yaxis_transform(), ha='right', va='bottom', fontsize=8)
    axis.text(0.99, 0.305, 'moderate', transform=axis.get_yaxis_transform(), ha='right', va='bottom', fontsize=8)
    axis.text(0.99, 0.505, 'large', transform=axis.get_yaxis_transform(), ha='right', va='bottom', fontsize=8)
    label_candidates = dataframe.assign(_label_score=dataframe['absolute_median_difference_um'] * dataframe['absolute_rank_biserial']).nlargest(label_n, '_label_score')
    for row in label_candidates.itertuples(index=False):
        axis.annotate(str(row.comparison), (float(row.absolute_median_difference_um), float(row.absolute_rank_biserial)), xytext=(4, 4), textcoords='offset points', fontsize=6.5)
    axis.set_xlabel('Absolute median difference in distance (µm)')
    axis.set_ylabel('Absolute rank-biserial effect size')
    axis.set_ylim(-0.02, 1.02)
    axis.set_title(title, loc='left', fontweight='bold')
    axis.legend(frameon=False, fontsize=8)
    hide_top_right_spines(axis)
    return save_figure(figure, output_stem)

def plot_ranked_horizontal(dataframe: pd.DataFrame, ranking_column: str, top_n: int, title: str, x_label: str, output_stem: Path) -> tuple[list[Path], pd.DataFrame]:
    ranked = dataframe.sort_values([ranking_column, 'absolute_median_difference_um', 'absolute_rank_biserial'], ascending=[False, False, False]).head(top_n).sort_values(ranking_column, ascending=True).copy()
    height = max(5.0, 0.31 * len(ranked) + 1.5)
    figure, axis = plt.subplots(figsize=(9.2, height))
    y = np.arange(len(ranked))
    values = ranked[ranking_column].to_numpy(dtype=float)
    axis.barh(y, values)
    axis.set_yticks(y)
    axis.set_yticklabels(ranked['comparison'], fontsize=7.5)
    for position, row in enumerate(ranked.itertuples(index=False)):
        annotation = f'Δmedian={row.median_difference_a_minus_b_um:+.2f} µm; r={row.rank_biserial_a_vs_b:+.3f}'
        axis.text(float(getattr(row, ranking_column)), position, '  ' + annotation, va='center', fontsize=6.6)
    maximum = float(np.nanmax(values))
    axis.set_xlim(0, maximum * 1.55 if maximum > 0 else 1)
    axis.set_xlabel(x_label)
    axis.set_title(title, loc='left', fontweight='bold')
    hide_top_right_spines(axis)
    paths = save_figure(figure, output_stem)
    return (paths, ranked.sort_values(ranking_column, ascending=False))

def plot_signed_difference_effect(dataframe: pd.DataFrame, top_n: int, title: str, output_stem: Path) -> tuple[list[Path], pd.DataFrame]:
    ranked = dataframe.sort_values(['absolute_median_difference_um', 'absolute_rank_biserial'], ascending=[False, False]).head(top_n).sort_values('median_difference_a_minus_b_um', ascending=True).copy()
    height = max(5.0, 0.34 * len(ranked) + 1.5)
    figure, axis = plt.subplots(figsize=(9.8, height))
    y = np.arange(len(ranked))
    signed_difference = ranked['median_difference_a_minus_b_um'].to_numpy(dtype=float)
    axis.axvline(0, linewidth=0.9)
    axis.hlines(y=y, xmin=0, xmax=signed_difference, linewidth=1.2)
    marker_sizes = 25 + 170 * ranked['absolute_rank_biserial'].to_numpy(dtype=float)
    axis.scatter(signed_difference, y, s=marker_sizes, alpha=0.72)
    axis.set_yticks(y)
    axis.set_yticklabels(ranked['comparison'], fontsize=7.5)
    for position, row in enumerate(ranked.itertuples(index=False)):
        horizontal_alignment = 'left' if row.median_difference_a_minus_b_um >= 0 else 'right'
        offset = 0.14 if row.median_difference_a_minus_b_um >= 0 else -0.14
        axis.text(row.median_difference_a_minus_b_um + offset, position, f'r={row.rank_biserial_a_vs_b:+.3f}', va='center', ha=horizontal_alignment, fontsize=6.8)
    axis.set_xlabel('Median difference: group A − group B (µm)\nMarker size represents |rank-biserial effect size|')
    axis.set_title(title, loc='left', fontweight='bold')
    hide_top_right_spines(axis)
    paths = save_figure(figure, output_stem)
    return (paths, ranked.sort_values('absolute_median_difference_um', ascending=False))

def plot_nt_effect_distributions(dataframe: pd.DataFrame, anatomical_level: str, output_stem: Path) -> list[Path] | None:
    nt_data = dataframe.loc[dataframe['anatomical_level'].eq(anatomical_level) & dataframe['nt_label'].ne('All contacts')].copy()
    if nt_data.empty:
        return None
    nt_order = sorted(nt_data['nt_label'].unique())
    values = [nt_data.loc[nt_data['nt_label'].eq(nt), 'absolute_rank_biserial'].to_numpy(dtype=float) for nt in nt_order]
    figure, axis = plt.subplots(figsize=(7.4, 4.8))
    axis.boxplot(values, labels=nt_order, showfliers=False, widths=0.65)
    axis.axhline(0.1, linewidth=0.8, linestyle='--')
    axis.axhline(0.3, linewidth=0.8, linestyle='--')
    axis.axhline(0.5, linewidth=0.8, linestyle='--')
    axis.set_ylim(-0.02, 1.02)
    axis.set_xlabel('Presynaptic neurotransmitter')
    axis.set_ylabel('Absolute rank-biserial effect size')
    level_label = 'subregions' if anatomical_level == 'subregion' else 'super-regions'
    axis.set_title(f'Effect-size distributions across {level_label}', loc='left', fontweight='bold')
    hide_top_right_spines(axis)
    return save_figure(figure, output_stem)

def analyse_family(dataframe: pd.DataFrame, family_label: str, output_dir: Path, top_n: int, label_n: int) -> list[dict]:
    output_dir.mkdir(parents=True, exist_ok=True)
    records = []
    scatter_paths = plot_effect_vs_difference(dataframe=dataframe, title=f'{family_label}: median difference versus effect size', output_stem=output_dir / 'median_difference_vs_effect_size', label_n=label_n)
    records.append({'figure': f'{family_label}: median difference versus effect size', 'files': ';'.join(map(str, scatter_paths))})
    difference_paths, top_difference = plot_ranked_horizontal(dataframe=dataframe, ranking_column='absolute_median_difference_um', top_n=top_n, title=f'{family_label}: largest median differences', x_label='Absolute median difference in distance (µm)', output_stem=output_dir / 'top_absolute_median_differences')
    top_difference.to_csv(output_dir / 'top_absolute_median_differences.csv', index=False)
    records.append({'figure': f'{family_label}: largest median differences', 'files': ';'.join(map(str, difference_paths))})
    effect_paths, top_effect = plot_ranked_horizontal(dataframe=dataframe, ranking_column='absolute_rank_biserial', top_n=top_n, title=f'{family_label}: largest rank-biserial effects', x_label='Absolute rank-biserial effect size', output_stem=output_dir / 'top_absolute_effect_sizes')
    top_effect.to_csv(output_dir / 'top_absolute_effect_sizes.csv', index=False)
    records.append({'figure': f'{family_label}: largest rank-biserial effects', 'files': ';'.join(map(str, effect_paths))})
    signed_paths, signed_table = plot_signed_difference_effect(dataframe=dataframe, top_n=top_n, title=f'{family_label}: direction and magnitude of strongest differences', output_stem=output_dir / 'top_signed_median_differences_with_effect_size')
    signed_table.to_csv(output_dir / 'top_signed_median_differences_with_effect_size.csv', index=False)
    records.append({'figure': f'{family_label}: signed differences and effect size', 'files': ';'.join(map(str, signed_paths))})
    return records

def make_effect_plots(pairwise_table, output_dir, top_n, label_n, significant_only, minimum_absolute_effect, minimum_absolute_median_difference_um) -> int:
    if top_n < 1:
        raise ValueError('--top-n must be at least 1.')
    if label_n < 0:
        raise ValueError('--label-n cannot be negative.')
    if not 0 <= minimum_absolute_effect <= 1:
        raise ValueError('--minimum-absolute-effect must be between 0 and 1.')
    if minimum_absolute_median_difference_um < 0:
        raise ValueError('--minimum-absolute-median-difference-um cannot be negative.')
    output_dir = output_dir.expanduser().resolve()
    table_dir = output_dir / 'tables'
    output_dir.mkdir(parents=True, exist_ok=True)
    table_dir.mkdir(parents=True, exist_ok=True)
    pairwise_path = require_file(Path(pairwise_table))
    pairwise = pd.read_csv(pairwise_path, low_memory=False)
    pairwise = validate_and_prepare(pairwise)
    before_exclusion = len(pairwise)
    group_a_normalized = pairwise['group_a'].astype(str).str.strip().str.upper()
    group_b_normalized = pairwise['group_b'].astype(str).str.strip().str.upper()
    excluded_normalized = {region.upper() for region in excluded_regions}
    exclude_mask = group_a_normalized.isin(excluded_normalized) | group_b_normalized.isin(excluded_normalized)
    pairwise = pairwise.loc[~exclude_mask].copy()
    removed_by_region_exclusion = before_exclusion - len(pairwise)
    if significant_only:
        pairwise = pairwise.loc[pairwise['significant_after_holm_0_05']].copy()
    pairwise = pairwise.loc[pairwise['absolute_rank_biserial'] >= minimum_absolute_effect].copy()
    pairwise = pairwise.loc[pairwise['absolute_median_difference_um'] >= minimum_absolute_median_difference_um].copy()
    if pairwise.empty:
        raise RuntimeError('No pairwise comparisons remain after filtering.')
    pairwise.to_csv(table_dir / 'pairwise_rows_used_for_plots.csv.gz', index=False, compression='gzip')
    figure_records = []
    family_definitions = [('All contacts — subregions', pairwise['anatomical_level'].eq('subregion') & pairwise['nt_label'].eq('All contacts'), output_dir / 'all_contacts_subregion'), ('All contacts — super-regions', pairwise['anatomical_level'].eq('super_region') & pairwise['nt_label'].eq('All contacts'), output_dir / 'all_contacts_super_region'), ('NT-specific — subregions', pairwise['anatomical_level'].eq('subregion') & pairwise['nt_label'].ne('All contacts'), output_dir / 'nt_subregion'), ('NT-specific — super-regions', pairwise['anatomical_level'].eq('super_region') & pairwise['nt_label'].ne('All contacts'), output_dir / 'nt_super_region')]
    for family_label, mask, family_output in family_definitions:
        family_data = pairwise.loc[mask].copy()
        if family_data.empty:
            continue
        figure_records.extend(analyse_family(dataframe=family_data, family_label=family_label, output_dir=family_output, top_n=min(top_n, len(family_data)), label_n=min(label_n, len(family_data))))
    for anatomical_level in ['subregion', 'super_region']:
        output_name = 'nt_effect_size_distribution_subregions' if anatomical_level == 'subregion' else 'nt_effect_size_distribution_super_regions'
        paths = plot_nt_effect_distributions(dataframe=pairwise, anatomical_level=anatomical_level, output_stem=output_dir / output_name)
        if paths is not None:
            figure_records.append({'figure': output_name, 'files': ';'.join(map(str, paths))})
    effect_summary = pairwise.groupby(['anatomical_level', 'nt_label'], as_index=False, dropna=False).agg(number_of_comparisons=('comparison', 'size'), median_absolute_median_difference_um=('absolute_median_difference_um', 'median'), maximum_absolute_median_difference_um=('absolute_median_difference_um', 'max'), median_absolute_rank_biserial=('absolute_rank_biserial', 'median'), maximum_absolute_rank_biserial=('absolute_rank_biserial', 'max'), proportion_small_or_larger=('absolute_rank_biserial', lambda values: float(np.mean(np.asarray(values) >= 0.1))), proportion_moderate_or_larger=('absolute_rank_biserial', lambda values: float(np.mean(np.asarray(values) >= 0.3))), proportion_large=('absolute_rank_biserial', lambda values: float(np.mean(np.asarray(values) >= 0.5))))
    effect_summary.to_csv(table_dir / 'effect_magnitude_summary_by_analysis.csv', index=False)
    pd.DataFrame(figure_records).to_csv(table_dir / 'figure_manifest.csv', index=False)
    return 0

# Figures 09-12
adv_metric = schematic_metric
adv_sig_threshold = sig_outline_threshold
adv_coverage_um = coverage_um
adv_coverage_xmax = coverage_xmax

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 13,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "svg.fonttype": "none",
})

adv_superregions = {
    "OL": ["ME_L", "ME_R", "AME_L", "AME_R", "LO_L", "LO_R", "LOP_L", "LOP_R"],
    "MB": ["MB_CA_R", "MB_CA_L", "MB_PED_L", "MB_PED_R", "MB_VL_R", "MB_VL_L", "MB_ML_R", "MB_ML_L"],
    "CX": ["FB", "EB", "PB", "NO"],
    "LX": ["BU_L", "BU_R", "LAL_R", "LAL_L", "GA_R", "GA_L"],
    "VLNP": ["AOTU_R", "AOTU_L", "PVLP_L", "AVLP_L", "PVLP_R", "AVLP_R", "PLP_L", "PLP_R", "WED_L", "WED_R"],
    "LH": ["LH_L", "LH_R"],
    "SNP": ["SLP_L", "SLP_R", "SIP_L", "SIP_R", "SMP_R", "SMP_L"],
    "INP": ["CRE_L", "CRE_R", "ICL_L", "SCL_L", "ICL_R", "SCL_R", "IB_L", "IB_R", "ATL_L", "ATL_R", "VES_L", "VES_R", "CAN_R", "CAN_L", "AMMC_R", "AMMC_L", "FLA_R", "FLA_L", "AL_R", "AL_L"],
    "AL": ["SAD", "PRW"],
    "VMNP": ["SPS_L", "SPS_R", "IPS_L", "IPS_R"],
    "PENP": ["EPA_R", "EPA_L", "GOR_L", "GOR_R"],
    "GNG": ["GNG"],
    "OCG": ["OCG"],
}

adv_super_order = list(adv_superregions.keys())
adv_region_to_super = {
    neuropil: super_region
    for super_region, neuropils in adv_superregions.items()
    for neuropil in neuropils
}

adv_super_pos = {
    "OL":   (0.07, 0.72),
    "VLNP": (0.22, 0.58),
    "LH":   (0.31, 0.66),
    "MB":   (0.42, 0.72),
    "CX":   (0.50, 0.56),
    "LX":   (0.36, 0.50),
    "SNP":  (0.60, 0.67),
    "INP":  (0.71, 0.56),
    "AL":   (0.23, 0.35),
    "VMNP": (0.52, 0.36),
    "PENP": (0.72, 0.36),
    "GNG":  (0.50, 0.18),
    "OCG":  (0.86, 0.18),
}

def save_svg_figure(fig: plt.Figure, stem: Path) -> Path:
    path = stem.with_suffix(".svg")
    fig.savefig(path, format="svg", bbox_inches="tight")
    return path

def load_figure_inputs(final_dir: Path, task3b_dir: Path):
    table_dir = final_dir / "FINAL_TABLES"

    if not table_dir.exists():
        table_dir = final_dir / "tables"

    if not table_dir.exists():
        raise FileNotFoundError(
            f"Could not find FINAL_TABLES or tables in {final_dir}"
        )

    all_super = pd.read_csv(
        table_dir / "02_all_contacts_superregion_exact_n_descriptives.csv"
    )
    nt_super = pd.read_csv(
        table_dir / "04_nt_superregion_exact_n_descriptives.csv"
    )
    pairwise = pd.read_csv(
        table_dir / "06_pairwise_mannwhitney_holm_all.csv.gz"
    )
    super_hist = pd.read_csv(
        task3b_dir / "neuropil_distance_histograms.csv.gz"
    )

    return all_super, nt_super, pairwise, super_hist

def standardize_superregion_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "super_region" in df.columns and "superregion" not in df.columns:
        df = df.rename(columns={"super_region": "superregion"})

    return df

def add_superregion_column(
    df: pd.DataFrame,
    region_col: str = "neuropil",
) -> pd.DataFrame:
    out = df.copy()
    out["superregion"] = out[region_col].map(adv_region_to_super)
    return out.loc[out["superregion"].notna()].copy()

def compute_effect_scores(pairwise: pd.DataFrame) -> pd.DataFrame:
    sub = pairwise.loc[
        pairwise["analysis_family"]
        == "all_contacts_across_superregions"
    ].copy()

    rows = []

    for _, row in sub.iterrows():
        group_a = row["group_a"]
        group_b = row["group_b"]
        effect = float(row["rank_biserial_a_vs_b"])

        rows.append(
            {
                "superregion": group_a,
                "effect_component": effect,
            }
        )
        rows.append(
            {
                "superregion": group_b,
                "effect_component": -effect,
            }
        )

    if not rows:
        return pd.DataFrame(
            columns=["superregion", "effect_score"]
        )

    return (
        pd.DataFrame(rows)
        .groupby("superregion", as_index=False)["effect_component"]
        .mean()
        .rename(columns={"effect_component": "effect_score"})
    )

def prepare_superregion_summary(
    all_super: pd.DataFrame,
    pairwise: pd.DataFrame,
) -> pd.DataFrame:
    df = standardize_superregion_columns(all_super)
    effects = compute_effect_scores(pairwise)

    df = df.merge(
        effects,
        on="superregion",
        how="left",
    )

    df["superregion"] = pd.Categorical(
        df["superregion"],
        categories=adv_super_order,
        ordered=True,
    )

    return df.sort_values("superregion").reset_index(drop=True)

def make_superregion_palette(labels: list[str]) -> dict:
    cmap = plt.get_cmap("tab20")
    return {
        label: cmap(index % 20)
        for index, label in enumerate(labels)
    }

def figure_09_overview(
    super_summary: pd.DataFrame,
    output_dir: Path,
) -> None:
    metric_col = (
        "sample_median_um"
        if adv_metric == "median_distance"
        else "effect_score"
    )
    metric_label = (
        "Median distance (µm)"
        if adv_metric == "median_distance"
        else "Effect score"
    )

    valid = super_summary.dropna(subset=[metric_col]).copy()
    norm = Normalize(
        valid[metric_col].min(),
        valid[metric_col].max(),
    )
    cmap = plt.get_cmap(
        "viridis"
        if adv_metric == "median_distance"
        else "coolwarm"
    )

    ranked = (
        super_summary
        .sort_values("sample_median_um", ascending=True)
        .copy()
    )
    ranked["y"] = np.arange(len(ranked))

    fig = plt.figure(figsize=(14.2, 6.8))
    grid = fig.add_gridspec(
        1,
        2,
        width_ratios=[1.0, 1.35],
        wspace=0.16,
    )

    ax_left = fig.add_subplot(grid[0, 0])
    ax_left.set_title(
        "a  Super-region overview",
        loc="left",
        fontweight="semibold",
        pad=10,
    )

    brain = patches.Ellipse(
        (0.5, 0.5),
        0.92,
        0.72,
        facecolor="0.97",
        edgecolor="0.78",
        linewidth=1.2,
    )
    ax_left.add_patch(brain)
    ax_left.plot(
        [0.5, 0.5],
        [0.16, 0.84],
        color="0.86",
        linewidth=0.8,
    )
    ax_left.plot(
        [0.12, 0.88],
        [0.5, 0.5],
        color="0.91",
        linewidth=0.7,
    )

    max_n = float(super_summary["n"].max())

    for _, row in super_summary.iterrows():
        superregion = str(row["superregion"])

        if superregion not in adv_super_pos:
            continue

        x, y = adv_super_pos[superregion]
        value = row[metric_col]

        color = (
            cmap(norm(value))
            if pd.notna(value)
            else (0.85, 0.85, 0.85, 1.0)
        )

        size = (
            1400
            + 700
            * (
                np.sqrt(float(row.get("n", 0)))
                / np.sqrt(max_n)
            )
        )

        ax_left.scatter(
            [x],
            [y],
            s=size,
            color=color,
            edgecolor="0.15",
            linewidth=0.8,
            zorder=3,
        )
        ax_left.text(
            x,
            y,
            superregion,
            ha="center",
            va="center",
            fontsize=12,
            fontweight="semibold",
            zorder=4,
        )

    scalar_map = ScalarMappable(norm=norm, cmap=cmap)
    scalar_map.set_array([])

    colorbar = fig.colorbar(
        scalar_map,
        ax=ax_left,
        fraction=0.046,
        pad=0.03,
    )
    colorbar.set_label(metric_label, fontsize=12)
    colorbar.ax.tick_params(labelsize=10)

    ax_left.set_xlim(0, 1)
    ax_left.set_ylim(0, 1)
    ax_left.axis("off")

    ax_right = fig.add_subplot(grid[0, 1])
    ax_right.set_title(
        "b  Synapse-to-trachea distance",
        loc="left",
        fontweight="semibold",
        pad=10,
    )

    ax_right.hlines(
        ranked["y"],
        ranked["sample_q1_um"],
        ranked["sample_q3_um"],
        color="0.72",
        linewidth=2.2,
        zorder=1,
    )
    ax_right.scatter(
        ranked["sample_median_um"],
        ranked["y"],
        s=68,
        color="#3F6F8F",
        edgecolor="white",
        linewidth=0.6,
        zorder=3,
    )

    ax_right.set_yticks(ranked["y"])
    ax_right.set_yticklabels(ranked["superregion"])
    ax_right.set_xlabel("Distance to nearest trachea (µm)")
    ax_right.set_ylabel("")
    ax_right.grid(
        axis="x",
        color="0.90",
        linewidth=0.7,
    )
    ax_right.set_axisbelow(True)
    ax_right.spines["top"].set_visible(False)
    ax_right.spines["right"].set_visible(False)

    fig.subplots_adjust(
        left=0.04,
        right=0.985,
        bottom=0.14,
        top=0.92,
    )

    save_svg_figure(
        fig,
        output_dir / "09_superregion_atlas_and_ranked",
    )
    plt.show()
    plt.close(fig)

    ranked[
        [
            "superregion",
            "n",
            "sample_median_um",
            "sample_q1_um",
            "sample_q3_um",
            "sample_iqr_um",
            "effect_score",
        ]
    ].to_csv(
        output_dir
        / "09_superregion_atlas_and_ranked_source.csv",
        index=False,
    )

def build_nt_outline_table(
    nt_super: pd.DataFrame,
    pairwise: pd.DataFrame,
) -> pd.DataFrame:
    nt_super = standardize_superregion_columns(nt_super)

    rows = []
    families = [
        f"{nt}_across_superregions"
        for nt in nt_order
    ]

    for nt, family in zip(nt_order, families):
        family_table = pairwise.loc[
            pairwise["analysis_family"] == family
        ].copy()

        possible = set(
            nt_super.loc[
                nt_super["nt_type"] == nt,
                "superregion",
            ].astype(str)
        )

        totals = {superregion: 0 for superregion in possible}
        significant = {
            superregion: 0
            for superregion in possible
        }

        for _, row in family_table.iterrows():
            group_a = str(row["group_a"])
            group_b = str(row["group_b"])

            for superregion in [group_a, group_b]:
                if superregion not in totals:
                    continue

                totals[superregion] += 1

                if bool(row["significant_after_holm_0_05"]):
                    significant[superregion] += 1

        for superregion in possible:
            fraction = (
                significant[superregion]
                / totals[superregion]
                if totals[superregion]
                else np.nan
            )

            rows.append(
                {
                    "nt_type": nt,
                    "superregion": superregion,
                    "significant_pair_fraction": fraction,
                    "robust_outline": (
                        fraction
                        >= adv_sig_threshold
                        if pd.notna(fraction)
                        else False
                    ),
                }
            )

    outline = pd.DataFrame(rows)

    return nt_super.merge(
        outline,
        on=["nt_type", "superregion"],
        how="left",
    )

def figure_10_nt_matrix(
    nt_super: pd.DataFrame,
    pairwise: pd.DataFrame,
    output_dir: Path,
) -> None:
    df = build_nt_outline_table(
        nt_super,
        pairwise,
    )

    df["superregion"] = pd.Categorical(
        df["superregion"],
        categories=adv_super_order,
        ordered=True,
    )
    df["nt_type"] = pd.Categorical(
        df["nt_type"],
        categories=nt_order,
        ordered=True,
    )

    df = (
        df.sort_values(["superregion", "nt_type"])
        .reset_index(drop=True)
    )

    max_n = float(df["n"].max())

    norm = Normalize(
        df["sample_median_um"].min(),
        df["sample_median_um"].max(),
    )
    cmap = plt.get_cmap("magma_r")

    fig, axis = plt.subplots(figsize=(9.6, 7.6))

    for row_index, superregion in enumerate(adv_super_order):
        subset = df.loc[
            df["superregion"] == superregion
        ]

        for column_index, nt_type in enumerate(nt_order):
            one = subset.loc[
                subset["nt_type"] == nt_type
            ]

            if one.empty:
                continue

            row = one.iloc[0]

            size = (
                55
                + 900
                * np.sqrt(float(row["n"]) / max_n)
            )

            outline = bool(
                row.get("robust_outline", False)
            )

            axis.scatter(
                column_index,
                row_index,
                s=size,
                color=cmap(
                    norm(row["sample_median_um"])
                ),
                edgecolor="0.10" if outline else "none",
                linewidth=1.7 if outline else 0.0,
            )

    axis.set_xticks(range(len(nt_order)))
    axis.set_xticklabels(nt_order)
    axis.set_yticks(range(len(adv_super_order)))
    axis.set_yticklabels(adv_super_order)

    axis.invert_yaxis()
    axis.set_xlim(-0.6, len(nt_order) - 0.4)
    axis.set_ylim(
        len(adv_super_order) - 0.4,
        -0.6,
    )

    axis.set_title(
        "NT × super-region",
        loc="left",
        fontweight="semibold",
        pad=10,
    )
    axis.set_xlabel("Neurotransmitter")
    axis.set_ylabel("Super-region")

    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

    scalar_map = ScalarMappable(
        norm=norm,
        cmap=cmap,
    )
    scalar_map.set_array([])

    colorbar = fig.colorbar(
        scalar_map,
        ax=axis,
        fraction=0.05,
        pad=0.03,
    )
    colorbar.set_label(
        "Median distance to nearest trachea (µm)"
    )
    colorbar.ax.tick_params(labelsize=10)

    size_handles = [
        axis.scatter(
            [],
            [],
            s=55 + 900 * np.sqrt(fraction),
            color="0.55",
            alpha=0.8,
            label=label,
        )
        for fraction, label in zip(
            [0.20, 0.50, 1.00],
            ["Low", "Medium", "High"],
        )
    ]

    outline_handle = axis.scatter(
        [],
        [],
        s=170,
        facecolor="white",
        edgecolor="0.10",
        linewidth=1.7,
        label="Robust",
    )

    axis.legend(
        handles=size_handles + [outline_handle],
        title="Relative contact count",
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.00),
        fontsize=11.5,
        title_fontsize=11.5,
        labelspacing=0.8,
    )

    fig.subplots_adjust(
        left=0.12,
        right=0.78,
        bottom=0.12,
        top=0.93,
    )

    save_svg_figure(
        fig,
        output_dir / "10_nt_superregion_dot_matrix",
    )
    plt.show()
    plt.close(fig)

    df[
        [
            "superregion",
            "nt_type",
            "n",
            "sample_median_um",
            "sample_q1_um",
            "sample_q3_um",
            "significant_pair_fraction",
            "robust_outline",
        ]
    ].to_csv(
        output_dir
        / "10_nt_superregion_dot_matrix_source.csv",
        index=False,
    )

def aggregate_hist_to_superregions(
    hist_df: pd.DataFrame,
) -> pd.DataFrame:
    df = add_superregion_column(
        hist_df,
        region_col="neuropil",
    )

    return (
        df.groupby(
            [
                "superregion",
                "bin_left_um",
                "bin_right_um",
            ],
            as_index=False,
        )["count"]
        .sum()
        .sort_values(
            ["superregion", "bin_left_um"]
        )
        .reset_index(drop=True)
    )

def coverage_summary_from_hist(
    super_hist: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    grouped = aggregate_hist_to_superregions(
        super_hist
    )

    curve_parts = []
    summary_rows = []

    for superregion, subset in grouped.groupby(
        "superregion",
        sort=False,
    ):
        subset = (
            subset.sort_values("bin_right_um")
            .copy()
        )

        total = subset["count"].sum()

        subset["cum_count"] = (
            subset["count"].cumsum()
        )
        subset["fraction"] = (
            subset["cum_count"] / total
        )
        subset["superregion"] = superregion

        curve_parts.append(subset)

        x = subset["bin_right_um"].to_numpy(float)
        y = subset["fraction"].to_numpy(float)

        r50 = (
            float(np.interp(0.50, y, x))
            if len(x)
            else np.nan
        )
        r90 = (
            float(np.interp(0.90, y, x))
            if len(x)
            else np.nan
        )

        row = {
            "superregion": superregion,
            "n": int(total),
            "r50_um": r50,
            "r90_um": r90,
        }

        for radius_um in adv_coverage_um:
            row[
                f"within_{radius_um:g}_um"
            ] = float(
                np.interp(
                    radius_um,
                    x,
                    y,
                    left=0.0,
                    right=1.0,
                )
            )

        summary_rows.append(row)

    curves = pd.concat(
        curve_parts,
        ignore_index=True,
    )

    summary = pd.DataFrame(summary_rows)

    summary["superregion"] = pd.Categorical(
        summary["superregion"],
        categories=adv_super_order,
        ordered=True,
    )

    summary = (
        summary.sort_values("superregion")
        .reset_index(drop=True)
    )

    return curves, summary

def figure_11_coverage(
    curves: pd.DataFrame,
    summary: pd.DataFrame,
    output_dir: Path,
) -> None:
    palette = make_superregion_palette(
        adv_super_order
    )

    fig = plt.figure(figsize=(14.2, 6.8))
    grid = fig.add_gridspec(
        1,
        2,
        width_ratios=[1.42, 1.0],
        wspace=0.25,
    )

    ax_left = fig.add_subplot(grid[0, 0])

    for superregion in adv_super_order:
        subset = curves.loc[
            curves["superregion"] == superregion
        ]

        if subset.empty:
            continue

        subset = subset.loc[
            subset["bin_right_um"] <= adv_coverage_xmax
        ]

        ax_left.plot(
            subset["bin_right_um"],
            subset["fraction"],
            linewidth=2.2,
            color=palette[superregion],
            label=superregion,
        )

    for threshold in adv_coverage_um:
        ax_left.axvline(
            threshold,
            linewidth=0.7,
            linestyle=(0, (2, 3)),
            color="0.82",
            zorder=0,
        )

    ax_left.set_title(
        "a  Distance-threshold coverage",
        loc="left",
        fontweight="semibold",
        pad=10,
    )
    ax_left.set_xlabel(
        "Distance to nearest trachea (µm)"
    )
    ax_left.set_ylabel(
        "Cumulative fraction of synapses"
    )
    ax_left.set_xlim(0, adv_coverage_xmax)
    ax_left.set_ylim(0, 1.01)
    ax_left.set_yticks(np.linspace(0, 1, 6))

    ax_left.spines["top"].set_visible(False)
    ax_left.spines["right"].set_visible(False)

    legend = ax_left.legend(
        frameon=False,
        ncol=2,
        loc="lower right",
        bbox_to_anchor=(0.99, 0.02),
        fontsize=11.5,
        columnspacing=1.1,
        handlelength=2.2,
        handletextpad=0.55,
        borderaxespad=0.2,
    )

    for line in legend.get_lines():
        line.set_linewidth(2.8)

    ax_right = fig.add_subplot(grid[0, 1])

    rank = (
        summary
        .dropna(subset=["r50_um", "r90_um"])
        .sort_values("r50_um", ascending=True)
        .reset_index(drop=True)
    )

    y = np.arange(len(rank))

    ax_right.hlines(
        y,
        rank["r50_um"],
        rank["r90_um"],
        color="0.72",
        linewidth=2.2,
        zorder=1,
    )

    ax_right.scatter(
        rank["r50_um"],
        y,
        s=72,
        facecolor="0.15",
        edgecolor="0.15",
        linewidth=0.9,
        label=r"$r_{50}$",
        zorder=3,
    )

    ax_right.scatter(
        rank["r90_um"],
        y,
        s=78,
        facecolor="white",
        edgecolor="0.15",
        linewidth=1.5,
        label=r"$r_{90}$",
        zorder=3,
    )

    ax_right.set_title(
        r"b  $r_{50}$ and $r_{90}$",
        loc="left",
        fontweight="semibold",
        pad=10,
    )
    ax_right.set_yticks(y)
    ax_right.set_yticklabels(
        rank["superregion"]
    )
    ax_right.set_xlabel(
        "Distance to nearest trachea (µm)"
    )
    ax_right.set_ylabel("")

    ax_right.grid(
        axis="x",
        color="0.90",
        linewidth=0.7,
    )
    ax_right.set_axisbelow(True)

    ax_right.spines["top"].set_visible(False)
    ax_right.spines["right"].set_visible(False)

    ax_right.legend(
        frameon=False,
        loc="lower right",
        fontsize=15,
        markerscale=1.25,
        handletextpad=0.65,
        labelspacing=0.6,
        borderaxespad=0.7,
    )

    fig.subplots_adjust(
        left=0.07,
        right=0.985,
        bottom=0.14,
        top=0.92,
    )

    save_svg_figure(
        fig,
        output_dir / "11_superregion_coverage_curves",
    )
    plt.show()
    plt.close(fig)

    summary.to_csv(
        output_dir
        / "11_superregion_coverage_summary.csv",
        index=False,
    )

    curves.to_csv(
        output_dir
        / "11_superregion_coverage_curve_points.csv.gz",
        index=False,
        compression="gzip",
    )

def cluster_order(
    matrix: pd.DataFrame,
    axis: int = 0,
) -> list:
    frame = (
        matrix.copy()
        if axis == 0
        else matrix.T.copy()
    )

    if frame.shape[0] <= 2:
        return list(frame.index)

    values = frame.to_numpy(dtype=float)

    if np.allclose(values, values[0]):
        return list(frame.index)

    distances = pdist(
        values,
        metric="euclidean",
    )

    if np.allclose(distances, 0):
        return list(frame.index)

    linkage_matrix = linkage(
        distances,
        method="average",
    )

    order = leaves_list(linkage_matrix)

    return list(frame.index[order])

def figure_12_fingerprint(
    nt_super: pd.DataFrame,
    output_dir: Path,
) -> None:
    df = standardize_superregion_columns(
        nt_super
    )

    median_matrix = (
        df.pivot_table(
            index="superregion",
            columns="nt_type",
            values="sample_median_um",
            aggfunc="first",
        )
        .reindex(
            index=adv_super_order,
            columns=nt_order,
        )
    )

    regional_baseline = median_matrix.median(
        axis=1
    )

    regional_sd = (
        median_matrix.std(axis=1)
        .replace(0, np.nan)
    )

    z_matrix = (
        median_matrix
        .sub(regional_baseline, axis=0)
        .div(regional_sd, axis=0)
        .fillna(0.0)
    )

    row_order = cluster_order(
        z_matrix,
        axis=0,
    )
    column_order = cluster_order(
        z_matrix,
        axis=1,
    )

    z_plot = z_matrix.loc[
        row_order,
        column_order,
    ]

    row_linkage = (
        linkage(
            pdist(
                z_plot.to_numpy(dtype=float),
                metric="euclidean",
            ),
            method="average",
        )
        if z_plot.shape[0] > 2
        else None
    )

    column_linkage = (
        linkage(
            pdist(
                z_plot.T.to_numpy(dtype=float),
                metric="euclidean",
            ),
            method="average",
        )
        if z_plot.shape[1] > 2
        else None
    )

    fig = plt.figure(figsize=(10.4, 9.0))

    grid = fig.add_gridspec(
        2,
        2,
        width_ratios=[0.22, 1.0],
        height_ratios=[0.22, 1.0],
        wspace=0.02,
        hspace=0.02,
    )

    ax_blank = fig.add_subplot(grid[0, 0])
    ax_blank.axis("off")

    ax_columns = fig.add_subplot(grid[0, 1])

    if column_linkage is not None:
        dendrogram(
            column_linkage,
            ax=ax_columns,
            no_labels=True,
            color_threshold=0,
            above_threshold_color="0.15",
        )

    ax_columns.axis("off")

    ax_rows = fig.add_subplot(grid[1, 0])

    if row_linkage is not None:
        dendrogram(
            row_linkage,
            ax=ax_rows,
            orientation="left",
            no_labels=True,
            color_threshold=0,
            above_threshold_color="0.15",
        )

    ax_rows.axis("off")

    axis = fig.add_subplot(grid[1, 1])

    image = axis.imshow(
        z_plot.to_numpy(dtype=float),
        aspect="auto",
        cmap="coolwarm",
        vmin=-2.5,
        vmax=2.5,
    )

    axis.set_xticks(
        range(len(column_order))
    )
    axis.set_xticklabels(column_order)

    axis.set_yticks(
        range(len(row_order))
    )
    axis.set_yticklabels(row_order)

    axis.set_title(
        "NT proximity fingerprint",
        loc="left",
        fontweight="semibold",
        pad=10,
    )
    axis.set_xlabel("Neurotransmitter")
    axis.set_ylabel("Super-region")

    colorbar = fig.colorbar(
        image,
        ax=axis,
        fraction=0.046,
        pad=0.025,
    )
    colorbar.set_label(
        "Within-region z score"
    )
    colorbar.ax.tick_params(labelsize=10)

    for row_index in range(z_plot.shape[0]):
        for column_index in range(
            z_plot.shape[1]
        ):
            value = z_plot.iloc[
                row_index,
                column_index,
            ]

            axis.text(
                column_index,
                row_index,
                f"{value:.1f}",
                ha="center",
                va="center",
                fontsize=10.5,
                color=(
                    "0.10"
                    if abs(value) < 1.3
                    else "white"
                ),
            )

    save_svg_figure(
        fig,
        output_dir
        / "12_superregion_nt_fingerprint_clustered",
    )
    plt.show()
    plt.close(fig)

    z_plot.reset_index().to_csv(
        output_dir
        / "12_superregion_nt_fingerprint_clustered_source.csv",
        index=False,
    )

    median_matrix.loc[
        row_order,
        column_order,
    ].reset_index().to_csv(
        output_dir
        / "12_superregion_nt_median_matrix_source.csv",
        index=False,
    )

# Run
# Synapse + NT tables -> contact summaries
join_nt(
    syn_csv, nt_csv, nt_dir,
    join_chunk, nt_min_score, join_sample_n,
)

# Synapse coordinates + tracheole nodes -> distance summaries and plot samples
measure_proximity(
    syn_csv, nt_csv, trach_csv, nt_dir, dist_dir,
    dist_chunk, workers, trach_step_nm, min_trach_nodes,
    nt_min_score, coord_mode, hist_max_um, hist_bin_um,
    sample_region_n, sample_region_nt_n, sample_overshoot,
    seed,
)

# Distance outputs -> statistics and Figures 01-08
summarize_results(
    dist_dir, final_dir, groups_per_page, min_sample_n,
    sample_super_n, sample_super_nt_n, plot_qmax,
    top_effects_n, seed,
)

# Pairwise statistics -> effect-size figures
pairwise_csv = final_dir / "FINAL_TABLES" / "06_pairwise_mannwhitney_holm_all.csv.gz"
make_effect_plots(
    pairwise_csv, effect_dir, effect_top_n, effect_label_n,
    effect_sig_only, min_abs_effect, min_abs_median_um,
)

# Final tables + distance histograms -> Figures 09-12
warnings.filterwarnings("ignore", category=FutureWarning)
fig_dir.mkdir(parents=True, exist_ok=True)
all_super, nt_super, pairwise, super_hist = load_figure_inputs(final_dir, dist_dir)
super_summary = prepare_superregion_summary(all_super, pairwise)
curves, coverage = coverage_summary_from_hist(super_hist)
figure_09_overview(super_summary, fig_dir)
figure_10_nt_matrix(nt_super, pairwise, fig_dir)
figure_11_coverage(curves, coverage, fig_dir)
figure_12_fingerprint(nt_super, fig_dir)